# ENARES 2024 CRS04 — Stage 03
## NB02 · Skip logic, dominios, analytical y evidencia
Cubre Issues #29–#31.

In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, hashlib, os
auth.authenticate_user(); drive.mount('/content/drive')
PROJECT_ID='enares-2024-crs04'; LOCATION='US'; EXPECTED_ROWS=18807
ROOT_DRIVE=Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
LOG_DIR=ROOT_DRIVE/'05Resultados'/'logs'/'stage03'; SQL_DIR=ROOT_DRIVE/'02SQL'; OUTPUT_DIR=ROOT_DRIVE/'04Outputs'; DOCS_DIR=ROOT_DRIVE/'docs'; R_DIR=ROOT_DRIVE/'03Scripts_R'
for d in [LOG_DIR,SQL_DIR,OUTPUT_DIR,DOCS_DIR,R_DIR]: d.mkdir(parents=True,exist_ok=True)
RUN_UTC=datetime.now(timezone.utc).isoformat(); client=bigquery.Client(project=PROJECT_ID,location=LOCATION)
display(client.query('SELECT CURRENT_DATE() AS fecha_actual').result().to_dataframe())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.4 MB/s eta 0:00:00
Mounted at /content/drive


,fecha_actual
0,2026-08-11


## 1. Verificar cleaned

In [2]:
t=client.get_table(f'{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents')
if t.num_rows!=EXPECTED_ROWS: raise RuntimeError('Cleaned inválida')

## 2. Skip map

In [3]:
skip_map=pd.DataFrame([
{'block':'VP_HOGAR','gateway_var':'C3P203','open_value':1,'dependent_vars':'C3P201_1..C3P201_11','recode':'SYSMIS->0','keep_null_when':'C3P203 IS NULL','module':'3.2'},
{'block':'VF_HOGAR','gateway_var':'C3P207','open_value':1,'dependent_vars':'C3P205_1..C3P205_7','recode':'SYSMIS->0','keep_null_when':'C3P207 IS NULL','module':'3.2'},
{'block':'VP_ESCUELA','gateway_var':'C3P225','open_value':1,'dependent_vars':'C3P223_1..C3P223_14','recode':'SYSMIS->0','keep_null_when':'C3P225 IS NULL','module':'3.3'},
{'block':'VF_ESCUELA','gateway_var':'C3P229','open_value':1,'dependent_vars':'C3P227_1..C3P227_10','recode':'SYSMIS->0','keep_null_when':'C3P229 IS NULL','module':'3.3'},
{'block':'VS_12M','gateway_var':'CONFIRMAR_PDF','open_value':1,'dependent_vars':'C4P248_1..C4P248_16','recode':'SYSMIS->0','keep_null_when':'gateway IS NULL','module':'3.4'}])
skip_map.to_csv(LOG_DIR/'stage3_skip_map.csv',index=False); display(skip_map)
client.load_table_from_dataframe(skip_map,f'{PROJECT_ID}.enares2024_crs04_outputs.stage3_skip_map',job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()

,block,gateway_var,open_value,dependent_vars,recode,keep_null_when,module
0,VP_HOGAR,C3P203,1,C3P201_1..C3P201_11,SYSMIS->0,C3P203 IS NULL,3.2
1,VF_HOGAR,C3P207,1,C3P205_1..C3P205_7,SYSMIS->0,C3P207 IS NULL,3.2
2,VP_ESCUELA,C3P225,1,C3P223_1..C3P223_14,SYSMIS->0,C3P225 IS NULL,3.3
3,VF_ESCUELA,C3P229,1,C3P227_1..C3P227_10,SYSMIS->0,C3P229 IS NULL,3.3
4,VS_12M,CONFIRMAR_PDF,1,C4P248_1..C4P248_16,SYSMIS->0,gateway IS NULL,3.4


LoadJob<project=enares-2024-crs04, location=US, id=2a0830c6-7b5f-42ed-a392-4383cd670b35>

## 3. Dominios

In [4]:
defs={'C3P301_1':[1,2,3],'C3P301_2':[1,2,3],'C3P301_3':[1,2,3],'C3P301_4':[1,2,3],'C3P301_5':[1,2,3],'C3P301_6':[1,2,3],'C3P128':[1,2,3,4],'C4P129':list(range(1,10))}
rvals=pd.DataFrame([{'variable':v,'value':x} for v,xs in defs.items() for x in xs]); sql='SELECT CAST(variable_name AS STRING) variable_name, CAST(value AS STRING) value FROM `'+PROJECT_ID+'.enares2024_crs04_raw.metadata_crs04_value_labels`'; labels=client.query(sql).result().to_dataframe(); valid=set(zip(labels['variable_name'],labels['value'])); rvals['value_exists']=[(v,str(x)) in valid for v,x in zip(rvals['variable'],rvals['value'])]; rvals.to_csv(LOG_DIR/'stage3_value_domain_check.csv',index=False); display(rvals)

,variable,value,value_exists
0,C3P301_1,1,False
1,C3P301_1,2,False
2,C3P301_1,3,False
3,C3P301_2,1,False
4,C3P301_2,2,False
5,C3P301_2,3,False
6,C3P301_3,1,False
7,C3P301_3,2,False
8,C3P301_3,3,False
9,C3P301_4,1,False


## 4. Analytical inicial

In [5]:
# 4. Crear analytical_crs04_adolescents

analytical_sql = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents`
AS

WITH base AS (
  SELECT *
  FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
),

recodes AS (
  SELECT
    *,

    CONCAT(
      CAST(ID AS STRING),
      '_',
      CAST(C3ANIO AS STRING),
      '_',
      CAST(TURNO AS STRING),
      '_',
      UPPER(TRIM(CAST(C3SECC AS STRING)))
    ) AS ID_AULA,

    CASE
      WHEN C3P128 = 1 THEN 1
      WHEN C3P128 IN (2, 3) THEN 3
      WHEN C3P128 = 4 THEN 4
      ELSE NULL
    END AS idiomaHogar,

    CASE
      WHEN C3P301_4 = 1 THEN 1
      WHEN C3P301_4 = 2 THEN 0
      WHEN C3P301_4 = 3 THEN NULL
      ELSE NULL
    END AS justifica_castigo_docente,

    CASE
      WHEN C3P301_5 = 1 THEN 1
      WHEN C3P301_5 = 2 THEN 0
      WHEN C3P301_5 = 3 THEN NULL
      ELSE NULL
    END AS justifica_castigo_parental

  FROM base
),

indicators AS (
  SELECT
    *,

    CASE
      WHEN justifica_castigo_docente IS NULL
       AND justifica_castigo_parental IS NULL
      THEN NULL

      WHEN justifica_castigo_docente = 1
        OR justifica_castigo_parental = 1
      THEN 1

      ELSE 0
    END AS justifica_al_menos_una

  FROM recodes
)

SELECT *
FROM indicators
"""

# Guardar SQL como evidencia
(SQL_DIR / "stage3_master.sql").write_text(
    analytical_sql,
    encoding="utf-8"
)

(SQL_DIR / "stage3_create_crs04_analytical.sql").write_text(
    analytical_sql,
    encoding="utf-8"
)

# Mostrar y ejecutar
print(analytical_sql)

client.query(analytical_sql).result()

print("analytical_crs04_adolescents creada correctamente")


CREATE OR REPLACE TABLE
  `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`
AS

WITH base AS (
  SELECT *
  FROM `enares-2024-crs04.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
),

recodes AS (
  SELECT
    *,

    CONCAT(
      CAST(ID AS STRING),
      '_',
      CAST(C3ANIO AS STRING),
      '_',
      CAST(TURNO AS STRING),
      '_',
      UPPER(TRIM(CAST(C3SECC AS STRING)))
    ) AS ID_AULA,

    CASE
      WHEN C3P128 = 1 THEN 1
      WHEN C3P128 IN (2, 3) THEN 3
      WHEN C3P128 = 4 THEN 4
      ELSE NULL
    END AS idiomaHogar,

    CASE
      WHEN C3P301_4 = 1 THEN 1
      WHEN C3P301_4 = 2 THEN 0
      WHEN C3P301_4 = 3 THEN NULL
      ELSE NULL
    END AS justifica_castigo_docente,

    CASE
      WHEN C3P301_5 = 1 THEN 1
      WHEN C3P301_5 = 2 THEN 0
      WHEN C3P301_5 = 3 THEN NULL
      ELSE NULL
    END AS justifica_castigo_parental

  FROM base
),

indicators AS (
  SELECT
    *,

    CASE
      WHEN justifica_castigo_docent

## 5. Validaciones

In [6]:
A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# 1. Validar filas, llaves únicas, nulos y duplicados
analytical_check_sql = f"""
WITH key_counts AS (
  SELECT
    ID,
    COLEGIAL_ID,
    COUNT(*) AS key_n
  FROM `{A}`
  GROUP BY ID, COLEGIAL_ID
)
SELECT
  (
    SELECT COUNT(*)
    FROM `{A}`
  ) AS rows_analytical,

  COUNT(*) AS distinct_keys,

  (
    SELECT COUNTIF(ID IS NULL)
    FROM `{A}`
  ) AS id_nulls,

  (
    SELECT COUNTIF(COLEGIAL_ID IS NULL)
    FROM `{A}`
  ) AS colegial_id_nulls,

  COALESCE(SUM(key_n - 1), 0) AS duplicated_key_rows

FROM key_counts
"""

chk = (
    client.query(analytical_check_sql)
    .result()
    .to_dataframe()
)

chk.to_csv(
    LOG_DIR / "stage3_analytical_validation.csv",
    index=False
)

display(chk)

r = chk.iloc[0]

if (
    r["rows_analytical"] != EXPECTED_ROWS
    or r["distinct_keys"] != EXPECTED_ROWS
    or r["id_nulls"] > 0
    or r["colegial_id_nulls"] > 0
    or r["duplicated_key_rows"] > 0
):
    raise RuntimeError(
        "Analytical inválida. Revisar stage3_analytical_validation.csv"
    )

print("Analytical validada: 18,807 filas y 18,807 llaves únicas.")


# 2. Validar dominio 0/1/NULL de los indicadores implementados
inds = [
    "justifica_castigo_docente",
    "justifica_castigo_parental",
    "justifica_al_menos_una",
]

bad_parts = []
distribution_parts = []

for v in inds:
    distribution_sql = f"""
    SELECT
      '{v}' AS indicador,
      CAST(`{v}` AS STRING) AS valor,
      COUNT(*) AS n
    FROM `{A}`
    GROUP BY `{v}`
    ORDER BY `{v}`
    """

    distribution_parts.append(
        client.query(distribution_sql)
        .result()
        .to_dataframe()
    )

    bad_sql = f"""
    SELECT
      '{v}' AS indicador,
      CAST(`{v}` AS STRING) AS valor,
      COUNT(*) AS n
    FROM `{A}`
    WHERE `{v}` IS NOT NULL
      AND `{v}` NOT IN (0, 1)
    GROUP BY `{v}`
    """

    bad_parts.append(
        client.query(bad_sql)
        .result()
        .to_dataframe()
    )

indicator_distribution = pd.concat(
    distribution_parts,
    ignore_index=True
)

bad = pd.concat(
    bad_parts,
    ignore_index=True
)

indicator_distribution.to_csv(
    LOG_DIR / "stage3_indicator_distribution.csv",
    index=False
)

bad.to_csv(
    LOG_DIR / "stage3_indicator_domain_validation.csv",
    index=False
)

display(indicator_distribution)
display(bad)

if len(bad) > 0:
    raise RuntimeError(
        "Hay indicadores con valores fuera de 0/1/NULL."
    )

print("Indicadores validados: solo contienen 0, 1 o NULL.")

,rows_analytical,distinct_keys,id_nulls,colegial_id_nulls,duplicated_key_rows
0,18807,18807,0,0,0


Analytical validada: 18,807 filas y 18,807 llaves únicas.


,indicador,valor,n
0,justifica_castigo_docente,None,180
1,justifica_castigo_docente,0,17466
2,justifica_castigo_docente,1,1161
3,justifica_castigo_parental,None,454
4,justifica_castigo_parental,0,9805
5,justifica_castigo_parental,1,8548
6,justifica_al_menos_una,None,34
7,justifica_al_menos_una,0,10033
8,justifica_al_menos_una,1,8740


,indicador,valor,n


Indicadores validados: solo contienen 0, 1 o NULL.


# 3.1 1

In [7]:
A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

derived_31 = [
    "etnicidad",
    "etnicidad1",
    "idiomaHogar",
    "DISCAPACIDAD",
    "tipo_hogar1",
]

columns_to_replace = [
    column
    for column in derived_31
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(f"`{column}`" for column in columns_to_replace)
        + ")"
    )
else:
    source_select = "*"

sql_31 = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
    {source_select},

    CASE
        WHEN C4P129 IN (1, 2) THEN 1
        WHEN C4P129 IN (3, 4) THEN 3
        WHEN C4P129 = 5 THEN 5
        WHEN C4P129 = 6 THEN 6
        WHEN C4P129 = 7 THEN 7
        WHEN C4P129 = 8 THEN 8
        WHEN C4P129 = 9 THEN 9
        ELSE NULL
    END AS etnicidad,

    CASE
        WHEN C4P129 IN (1, 2) THEN 1
        WHEN C4P129 IN (3, 4) THEN 3
        WHEN C4P129 = 5 THEN 5
        WHEN C4P129 IN (6, 7, 8) THEN 6
        WHEN C4P129 = 9 THEN 9
        ELSE NULL
    END AS etnicidad1,

    CASE
        WHEN C3P128 = 1 THEN 1
        WHEN C3P128 IN (2, 3) THEN 3
        WHEN C3P128 = 4 THEN 4
        ELSE -1
    END AS idiomaHogar,

    CASE
        WHEN C4P130_1 = 1
          OR C4P130_2 = 1
          OR C4P130_3 = 1
          OR C4P130_4 = 1
          OR C4P130_5 = 1
          OR C4P130_6 = 1
        THEN 1

        WHEN C4P130_1 = 2
          AND C4P130_2 = 2
          AND C4P130_3 = 2
          AND C4P130_4 = 2
          AND C4P130_5 = 2
          AND C4P130_6 = 2
        THEN 0

        ELSE NULL
    END AS DISCAPACIDAD,

    CASE
        WHEN C3P105 = 2 THEN 3

        WHEN C3P115_1 IS NULL
          OR C3P115_2 IS NULL
          OR C3P115_3 IS NULL
          OR C3P115_4 IS NULL
        THEN -1

        ELSE NULL
    END AS tipo_hogar1

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_31_desagregaciones.sql").write_text(
    sql_31,
    encoding="utf-8"
)

print(sql_31)

client.query(sql_31).result()

print("Variables base de la sintaxis 3.1 creadas correctamente.")


CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
    * EXCEPT(`idiomaHogar`),

    CASE
        WHEN C4P129 IN (1, 2) THEN 1
        WHEN C4P129 IN (3, 4) THEN 3
        WHEN C4P129 = 5 THEN 5
        WHEN C4P129 = 6 THEN 6
        WHEN C4P129 = 7 THEN 7
        WHEN C4P129 = 8 THEN 8
        WHEN C4P129 = 9 THEN 9
        ELSE NULL
    END AS etnicidad,

    CASE
        WHEN C4P129 IN (1, 2) THEN 1
        WHEN C4P129 IN (3, 4) THEN 3
        WHEN C4P129 = 5 THEN 5
        WHEN C4P129 IN (6, 7, 8) THEN 6
        WHEN C4P129 = 9 THEN 9
        ELSE NULL
    END AS etnicidad1,

    CASE
        WHEN C3P128 = 1 THEN 1
        WHEN C3P128 IN (2, 3) THEN 3
        WHEN C3P128 = 4 THEN 4
        ELSE -1
    END AS idiomaHogar,

    CASE
        WHEN C4P130_1 = 1
          OR C4P130_2 = 1
          OR C4P130_3 = 1
          OR C4P130_4 = 1
          OR C4P130_5 = 1
          OR C4P130_6 = 1
        THEN 1

        WHEN C4P130_1

#  3.1 2

In [8]:
# ============================================================
# SPSS 3.1 — Actitudes y percepciones C3P301_1 a C3P301_6
# Fuente: 07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps
# Compatible con BigQuery Sandbox: CREATE OR REPLACE, sin UPDATE
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_301 = [f"C3P301_{i}" for i in range(1, 7)]

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_301 = sorted(set(required_301) - existing_columns)

if missing_301:
    raise RuntimeError(
        "No se puede ejecutar el bloque SPSS 3.1. "
        "Faltan variables fuente: "
        + ", ".join(missing_301)
    )

print("Variables C3P301_1 a C3P301_6 verificadas.")


# ------------------------------------------------------------
# 2. Permitir reejecución sin duplicar columnas
# ------------------------------------------------------------

derived_301 = [
    "c3p301_1_bin",
    "c3p301_2_bin",
    "c3p301_3_bin",
    "c3p301_4_bin",
    "c3p301_5_bin",
    "c3p301_6_bin",
    "justifica_castigo_docente",
    "justifica_castigo_parental",
    "justifica_al_menos_una",
]

columns_to_replace = [
    column
    for column in derived_301
    if column in existing_columns
]

if columns_to_replace:
    except_clause = (
        "* EXCEPT("
        + ", ".join(f"`{column}`" for column in columns_to_replace)
        + ")"
    )
else:
    except_clause = "*"


# ------------------------------------------------------------
# 3. Traducir RECODE SPSS -> CASE BigQuery
# ------------------------------------------------------------

sql_31_actitudes = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH source AS (
  SELECT
    {except_clause}
  FROM `{A}`
),

recodes AS (
  SELECT
    *,

    -- C3P301_1: recodificación invertida
    CASE
      WHEN C3P301_1 = 1 THEN 0
      WHEN C3P301_1 = 2 THEN 1
      WHEN C3P301_1 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_1_bin,

    -- Recodificación normal: 1=1, 2=0, 3=NULL
    CASE
      WHEN C3P301_2 = 1 THEN 1
      WHEN C3P301_2 = 2 THEN 0
      WHEN C3P301_2 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_2_bin,

    -- C3P301_3: recodificación invertida
    CASE
      WHEN C3P301_3 = 1 THEN 0
      WHEN C3P301_3 = 2 THEN 1
      WHEN C3P301_3 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_3_bin,

    CASE
      WHEN C3P301_4 = 1 THEN 1
      WHEN C3P301_4 = 2 THEN 0
      WHEN C3P301_4 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_4_bin,

    CASE
      WHEN C3P301_5 = 1 THEN 1
      WHEN C3P301_5 = 2 THEN 0
      WHEN C3P301_5 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_5_bin,

    CASE
      WHEN C3P301_6 = 1 THEN 1
      WHEN C3P301_6 = 2 THEN 0
      WHEN C3P301_6 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_6_bin

  FROM source
),

indicators AS (
  SELECT
    *,

    c3p301_4_bin AS justifica_castigo_docente,
    c3p301_5_bin AS justifica_castigo_parental,

    CASE
      WHEN c3p301_4_bin IS NULL
       AND c3p301_5_bin IS NULL
      THEN NULL

      WHEN c3p301_4_bin = 1
        OR c3p301_5_bin = 1
      THEN 1

      ELSE 0
    END AS justifica_al_menos_una

  FROM recodes
)

SELECT *
FROM indicators
"""

(SQL_DIR / "stage3_syntax_31_actitudes.sql").write_text(
    sql_31_actitudes,
    encoding="utf-8"
)

print(sql_31_actitudes)

client.query(sql_31_actitudes).result()

print("Bloque SPSS 3.1 de actitudes creado correctamente.")


# ------------------------------------------------------------
# 4. Validar dominios
# ------------------------------------------------------------

vars_31 = [
    "c3p301_1_bin",
    "c3p301_2_bin",
    "c3p301_3_bin",
    "c3p301_4_bin",
    "c3p301_5_bin",
    "c3p301_6_bin",
    "justifica_castigo_docente",
    "justifica_castigo_parental",
    "justifica_al_menos_una",
]

validation_parts = []

for variable in vars_31:
    validation_sql = f"""
    SELECT
      '{variable}' AS variable,
      COUNTIF(
        `{variable}` IS NOT NULL
        AND `{variable}` NOT IN (0, 1)
      ) AS invalid_values,
      COUNTIF(`{variable}` IS NULL) AS null_values,
      COUNTIF(`{variable}` = 0) AS zero_values,
      COUNTIF(`{variable}` = 1) AS one_values
    FROM `{A}`
    """

    validation_parts.append(
        client.query(validation_sql)
        .result()
        .to_dataframe()
    )

validation_31 = pd.concat(
    validation_parts,
    ignore_index=True
)

validation_31.to_csv(
    LOG_DIR / "stage3_syntax_31_actitudes_validation.csv",
    index=False
)

display(validation_31)

if (validation_31["invalid_values"] > 0).any():
    raise RuntimeError(
        "El bloque 3.1 contiene valores fuera de 0/1/NULL."
    )

print("Bloque SPSS 3.1 validado: todos los valores son 0, 1 o NULL.")

Variables C3P301_1 a C3P301_6 verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

WITH source AS (
  SELECT
    * EXCEPT(`justifica_castigo_docente`, `justifica_castigo_parental`, `justifica_al_menos_una`)
  FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`
),

recodes AS (
  SELECT
    *,

    -- C3P301_1: recodificación invertida
    CASE
      WHEN C3P301_1 = 1 THEN 0
      WHEN C3P301_1 = 2 THEN 1
      WHEN C3P301_1 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_1_bin,

    -- Recodificación normal: 1=1, 2=0, 3=NULL
    CASE
      WHEN C3P301_2 = 1 THEN 1
      WHEN C3P301_2 = 2 THEN 0
      WHEN C3P301_2 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_2_bin,

    -- C3P301_3: recodificación invertida
    CASE
      WHEN C3P301_3 = 1 THEN 0
      WHEN C3P301_3 = 2 THEN 1
      WHEN C3P301_3 = 3 THEN NULL
      ELSE NULL
    END AS c3p301_3_bin,

    CASE
      WHEN C3P301_4 = 1 THEN

,variable,invalid_values,null_values,zero_values,one_values
0,c3p301_1_bin,0,310,5972,12525
1,c3p301_2_bin,0,93,1059,17655
2,c3p301_3_bin,0,226,2698,15883
3,c3p301_4_bin,0,180,17466,1161
4,c3p301_5_bin,0,454,9805,8548
5,c3p301_6_bin,0,238,1131,17438
6,justifica_castigo_docente,0,180,17466,1161
7,justifica_castigo_parental,0,454,9805,8548
8,justifica_al_menos_una,0,34,10033,8740


Bloque SPSS 3.1 validado: todos los valores son 0, 1 o NULL.


#3.1.3

In [9]:
# ============================================================
# SPSS 3.1 — Reconocimiento de derechos
# Fuente:
# 07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps
# Compatible con BigQuery Sandbox: CREATE OR REPLACE, sin UPDATE
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_derechos = [
    "C3P301_1",
    "C3P301_2",
    "C3P301_3",
    "C3P301_6",
]

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_derechos = sorted(
    set(required_derechos) - existing_columns
)

if missing_derechos:
    raise RuntimeError(
        "No se puede crear el bloque de derechos. "
        "Faltan variables fuente: "
        + ", ".join(missing_derechos)
    )

print("Variables fuente del bloque de derechos verificadas.")


# ------------------------------------------------------------
# 2. Permitir reejecución sin duplicar columnas
# ------------------------------------------------------------

derived_derechos = [
    "reconoce_derecho_opinar",
    "reconoce_derecho_denunciar",
    "rechaza_dejar_estudiar",
    "rechaza_trabajo_infantil_necesidad",
    "indice_derechos",
    "reconoce_todos_derechos_clave",
    "reconoce_3omas_derechos",
]

columns_to_replace = [
    column
    for column in derived_derechos
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(f"`{column}`" for column in columns_to_replace)
        + ")"
    )
else:
    source_select = "*"


# ------------------------------------------------------------
# 3. Traducir SPSS a BigQuery SQL
# ------------------------------------------------------------

sql_31_derechos = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH source AS (
  SELECT
    {source_select}
  FROM `{A}`
),

components AS (
  SELECT
    *,

    -- RECODE C3P301_2 (1=1) (2=0) (3=SYSMIS)
    CASE
      WHEN C3P301_2 = 1 THEN 1
      WHEN C3P301_2 = 2 THEN 0
      WHEN C3P301_2 = 3 THEN NULL
      ELSE NULL
    END AS reconoce_derecho_opinar,

    -- RECODE C3P301_6 (1=1) (2=0) (3=SYSMIS)
    CASE
      WHEN C3P301_6 = 1 THEN 1
      WHEN C3P301_6 = 2 THEN 0
      WHEN C3P301_6 = 3 THEN NULL
      ELSE NULL
    END AS reconoce_derecho_denunciar,

    -- RECODE invertido: desacuerdo = reconocimiento del derecho
    -- C3P301_3 (1=0) (2=1) (3=SYSMIS)
    CASE
      WHEN C3P301_3 = 1 THEN 0
      WHEN C3P301_3 = 2 THEN 1
      WHEN C3P301_3 = 3 THEN NULL
      ELSE NULL
    END AS rechaza_dejar_estudiar,

    -- C3P301_1 (1=0) (2=1) (3=SYSMIS)
    CASE
      WHEN C3P301_1 = 1 THEN 0
      WHEN C3P301_1 = 2 THEN 1
      WHEN C3P301_1 = 3 THEN NULL
      ELSE NULL
    END AS rechaza_trabajo_infantil_necesidad

  FROM source
),

rights_index AS (
  SELECT
    *,

    -- SPSS SUM ignora componentes missing.
    -- Solo queda NULL cuando los cuatro componentes son missing.
    CASE
      WHEN reconoce_derecho_opinar IS NULL
       AND reconoce_derecho_denunciar IS NULL
       AND rechaza_dejar_estudiar IS NULL
       AND rechaza_trabajo_infantil_necesidad IS NULL
      THEN NULL

      ELSE
        COALESCE(reconoce_derecho_opinar, 0)
        + COALESCE(reconoce_derecho_denunciar, 0)
        + COALESCE(rechaza_dejar_estudiar, 0)
        + COALESCE(rechaza_trabajo_infantil_necesidad, 0)
    END AS indice_derechos

  FROM components
),

final AS (
  SELECT
    *,

    -- SPSS exige que los cuatro componentes tengan información válida.
    CASE
      WHEN reconoce_derecho_opinar IS NULL
        OR reconoce_derecho_denunciar IS NULL
        OR rechaza_dejar_estudiar IS NULL
        OR rechaza_trabajo_infantil_necesidad IS NULL
      THEN NULL

      WHEN reconoce_derecho_opinar = 1
       AND reconoce_derecho_denunciar = 1
       AND rechaza_dejar_estudiar = 1
       AND rechaza_trabajo_infantil_necesidad = 1
      THEN 1

      ELSE 0
    END AS reconoce_todos_derechos_clave,

    CASE
      WHEN indice_derechos IS NULL THEN NULL
      WHEN indice_derechos >= 3 THEN 1
      ELSE 0
    END AS reconoce_3omas_derechos

  FROM rights_index
)

SELECT *
FROM final
"""

# Guardar SQL como evidencia
(SQL_DIR / "stage3_syntax_31_derechos.sql").write_text(
    sql_31_derechos,
    encoding="utf-8"
)

print(sql_31_derechos)

client.query(sql_31_derechos).result()

print("Bloque SPSS 3.1 de reconocimiento de derechos creado.")


# ------------------------------------------------------------
# 4. Validar dominios e índice
# ------------------------------------------------------------

binary_derechos = [
    "reconoce_derecho_opinar",
    "reconoce_derecho_denunciar",
    "rechaza_dejar_estudiar",
    "rechaza_trabajo_infantil_necesidad",
    "reconoce_todos_derechos_clave",
    "reconoce_3omas_derechos",
]

validation_parts = []

for variable in binary_derechos:
    validation_sql = f"""
    SELECT
      '{variable}' AS variable,
      COUNTIF(
        `{variable}` IS NOT NULL
        AND `{variable}` NOT IN (0, 1)
      ) AS invalid_values,
      COUNTIF(`{variable}` IS NULL) AS null_values,
      COUNTIF(`{variable}` = 0) AS zero_values,
      COUNTIF(`{variable}` = 1) AS one_values
    FROM `{A}`
    """

    validation_parts.append(
        client.query(validation_sql)
        .result()
        .to_dataframe()
    )

validation_derechos = pd.concat(
    validation_parts,
    ignore_index=True
)

index_validation = client.query(f"""
SELECT
  COUNTIF(
    indice_derechos IS NOT NULL
    AND indice_derechos NOT BETWEEN 0 AND 4
  ) AS invalid_index_values,

  COUNTIF(
    reconoce_3omas_derechos = 1
    AND indice_derechos < 3
  ) AS inconsistent_three_or_more,

  COUNTIF(
    reconoce_todos_derechos_clave = 1
    AND indice_derechos != 4
  ) AS inconsistent_all_rights

FROM `{A}`
""").result().to_dataframe()

validation_derechos.to_csv(
    LOG_DIR / "stage3_syntax_31_derechos_binary_validation.csv",
    index=False
)

index_validation.to_csv(
    LOG_DIR / "stage3_syntax_31_derechos_index_validation.csv",
    index=False
)

display(validation_derechos)
display(index_validation)

if (validation_derechos["invalid_values"] > 0).any():
    raise RuntimeError(
        "Hay indicadores de derechos fuera de 0/1/NULL."
    )

if (index_validation.iloc[0] > 0).any():
    raise RuntimeError(
        "Falló la consistencia del índice de derechos."
    )

print(
    "Bloque de derechos validado: binarios 0/1/NULL "
    "e indice_derechos entre 0 y 4."
)

Variables fuente del bloque de derechos verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

WITH source AS (
  SELECT
    *
  FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`
),

components AS (
  SELECT
    *,

    -- RECODE C3P301_2 (1=1) (2=0) (3=SYSMIS)
    CASE
      WHEN C3P301_2 = 1 THEN 1
      WHEN C3P301_2 = 2 THEN 0
      WHEN C3P301_2 = 3 THEN NULL
      ELSE NULL
    END AS reconoce_derecho_opinar,

    -- RECODE C3P301_6 (1=1) (2=0) (3=SYSMIS)
    CASE
      WHEN C3P301_6 = 1 THEN 1
      WHEN C3P301_6 = 2 THEN 0
      WHEN C3P301_6 = 3 THEN NULL
      ELSE NULL
    END AS reconoce_derecho_denunciar,

    -- RECODE invertido: desacuerdo = reconocimiento del derecho
    -- C3P301_3 (1=0) (2=1) (3=SYSMIS)
    CASE
      WHEN C3P301_3 = 1 THEN 0
      WHEN C3P301_3 = 2 THEN 1
      WHEN C3P301_3 = 3 THEN NULL
      ELSE NULL
    END AS rechaza_dejar_estudiar,

    -- C3P301_1 

,variable,invalid_values,null_values,zero_values,one_values
0,reconoce_derecho_opinar,0,93,1059,17655
1,reconoce_derecho_denunciar,0,238,1131,17438
2,rechaza_dejar_estudiar,0,226,2698,15883
3,rechaza_trabajo_infantil_necesidad,0,310,5972,12525
4,reconoce_todos_derechos_clave,0,771,8222,9814
5,reconoce_3omas_derechos,0,2,2343,16462


,invalid_index_values,inconsistent_three_or_more,inconsistent_all_rights
0,0,0,0


Bloque de derechos validado: binarios 0/1/NULL e indice_derechos entre 0 y 4.


In [10]:
# ============================================================
# SPSS 3.1 — Roles de género y división del trabajo del hogar
# Fuente:
# 07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps
#
# Traduce:
# - C3P302_1..C3P302_7  -> grupos mujer/hombre/entrevistada-o
# - C3P302_8..C3P302_10 -> grupos mujer/hombre/nadie
# - Indicadores binarios por tarea
# - Conteos y proporciones
# - predominio_femenino_tareas
#
# Compatible con BigQuery Sandbox: CREATE OR REPLACE, sin UPDATE
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_roles = [
    f"C3P302_{i}"
    for i in range(1, 11)
]

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_roles = sorted(
    set(required_roles) - existing_columns
)

if missing_roles:
    raise RuntimeError(
        "No se puede crear el bloque de roles de género. "
        "Faltan variables fuente: "
        + ", ".join(missing_roles)
    )

print("Variables C3P302_1 a C3P302_10 verificadas.")


# ------------------------------------------------------------
# 2. Columnas derivadas que crea este bloque
# ------------------------------------------------------------

derived_roles = (
    [f"C3P302_{i}_gr3" for i in range(1, 8)]
    + [f"C3P302_{i}_gr4" for i in range(8, 11)]
    + [f"tarea{i}_fem" for i in range(1, 11)]
    + [f"tarea{i}_masc" for i in range(1, 11)]
    + [f"tarea{i}_nna" for i in range(1, 8)]
    + [f"tarea{i}_nadie" for i in range(8, 11)]
    + [
        "n_tareas_femeninas",
        "n_tareas_masculinas",
        "n_tareas_nna",
        "n_tareas_nadie",
        "n_tareas_validas_1_7",
        "n_tareas_validas_8_10",
        "n_tareas_validas_p302",
        "prop_tareas_femeninas",
        "prop_tareas_masculinas",
        "prop_tareas_nna",
        "prop_tareas_nadie",
        "predominio_femenino_tareas",
    ]
)

columns_to_replace = [
    column
    for column in derived_roles
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(
            f"`{column}`"
            for column in columns_to_replace
        )
        + ")"
    )
else:
    source_select = "*"


# ------------------------------------------------------------
# 3. Generar expresiones SQL repetitivas
# ------------------------------------------------------------

# P302.1 a P302.7:
# 2,4,6 -> mujer (1)
# 3,5,7 -> hombre (2)
# 1     -> entrevistada/o (3)

recodes_1_7 = []

for i in range(1, 8):
    recodes_1_7.append(
        f"""
        CASE
          WHEN C3P302_{i} IN (2, 4, 6) THEN 1
          WHEN C3P302_{i} IN (3, 5, 7) THEN 2
          WHEN C3P302_{i} = 1 THEN 3
          ELSE NULL
        END AS C3P302_{i}_gr3
        """
    )

# P302.8 a P302.10:
# 2,4,6 -> mujer (1)
# 3,5,7 -> hombre (2)
# 8     -> nadie (4)

recodes_8_10 = []

for i in range(8, 11):
    recodes_8_10.append(
        f"""
        CASE
          WHEN C3P302_{i} IN (2, 4, 6) THEN 1
          WHEN C3P302_{i} IN (3, 5, 7) THEN 2
          WHEN C3P302_{i} = 8 THEN 4
          ELSE NULL
        END AS C3P302_{i}_gr4
        """
    )

recode_expressions = ",\n".join(
    recodes_1_7 + recodes_8_10
)


# Indicadores binarios para tareas 1 a 7.
# Cuando la variable agrupada es missing, SPSS produce SYSMIS.

binary_1_7 = []

for i in range(1, 8):
    binary_1_7.extend([
        f"""
        CASE
          WHEN C3P302_{i}_gr3 IS NULL THEN NULL
          WHEN C3P302_{i}_gr3 = 1 THEN 1
          ELSE 0
        END AS tarea{i}_fem
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr3 IS NULL THEN NULL
          WHEN C3P302_{i}_gr3 = 2 THEN 1
          ELSE 0
        END AS tarea{i}_masc
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr3 IS NULL THEN NULL
          WHEN C3P302_{i}_gr3 = 3 THEN 1
          ELSE 0
        END AS tarea{i}_nna
        """,
    ])

# Indicadores binarios para tareas 8 a 10.

binary_8_10 = []

for i in range(8, 11):
    binary_8_10.extend([
        f"""
        CASE
          WHEN C3P302_{i}_gr4 IS NULL THEN NULL
          WHEN C3P302_{i}_gr4 = 1 THEN 1
          ELSE 0
        END AS tarea{i}_fem
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr4 IS NULL THEN NULL
          WHEN C3P302_{i}_gr4 = 2 THEN 1
          ELSE 0
        END AS tarea{i}_masc
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr4 IS NULL THEN NULL
          WHEN C3P302_{i}_gr4 = 4 THEN 1
          ELSE 0
        END AS tarea{i}_nadie
        """,
    ])

binary_expressions = ",\n".join(
    binary_1_7 + binary_8_10
)


# ------------------------------------------------------------
# 4. SQL principal
# ------------------------------------------------------------

sql_31_roles = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH source AS (
  SELECT
    {source_select}
  FROM `{A}`
),

grouped_tasks AS (
  SELECT
    *,
    {recode_expressions}
  FROM source
),

binary_tasks AS (
  SELECT
    *,
    {binary_expressions}
  FROM grouped_tasks
),

task_counts AS (
  SELECT
    *,

    -- SPSS SUM ignora componentes missing.
    -- Si todos están missing, el resultado queda NULL.

    CASE
      WHEN
        tarea1_fem IS NULL
        AND tarea2_fem IS NULL
        AND tarea3_fem IS NULL
        AND tarea4_fem IS NULL
        AND tarea5_fem IS NULL
        AND tarea6_fem IS NULL
        AND tarea7_fem IS NULL
        AND tarea8_fem IS NULL
        AND tarea9_fem IS NULL
        AND tarea10_fem IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea1_fem, 0)
        + COALESCE(tarea2_fem, 0)
        + COALESCE(tarea3_fem, 0)
        + COALESCE(tarea4_fem, 0)
        + COALESCE(tarea5_fem, 0)
        + COALESCE(tarea6_fem, 0)
        + COALESCE(tarea7_fem, 0)
        + COALESCE(tarea8_fem, 0)
        + COALESCE(tarea9_fem, 0)
        + COALESCE(tarea10_fem, 0)
    END AS n_tareas_femeninas,

    CASE
      WHEN
        tarea1_masc IS NULL
        AND tarea2_masc IS NULL
        AND tarea3_masc IS NULL
        AND tarea4_masc IS NULL
        AND tarea5_masc IS NULL
        AND tarea6_masc IS NULL
        AND tarea7_masc IS NULL
        AND tarea8_masc IS NULL
        AND tarea9_masc IS NULL
        AND tarea10_masc IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea1_masc, 0)
        + COALESCE(tarea2_masc, 0)
        + COALESCE(tarea3_masc, 0)
        + COALESCE(tarea4_masc, 0)
        + COALESCE(tarea5_masc, 0)
        + COALESCE(tarea6_masc, 0)
        + COALESCE(tarea7_masc, 0)
        + COALESCE(tarea8_masc, 0)
        + COALESCE(tarea9_masc, 0)
        + COALESCE(tarea10_masc, 0)
    END AS n_tareas_masculinas,

    CASE
      WHEN
        tarea1_nna IS NULL
        AND tarea2_nna IS NULL
        AND tarea3_nna IS NULL
        AND tarea4_nna IS NULL
        AND tarea5_nna IS NULL
        AND tarea6_nna IS NULL
        AND tarea7_nna IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea1_nna, 0)
        + COALESCE(tarea2_nna, 0)
        + COALESCE(tarea3_nna, 0)
        + COALESCE(tarea4_nna, 0)
        + COALESCE(tarea5_nna, 0)
        + COALESCE(tarea6_nna, 0)
        + COALESCE(tarea7_nna, 0)
    END AS n_tareas_nna,

    CASE
      WHEN
        tarea8_nadie IS NULL
        AND tarea9_nadie IS NULL
        AND tarea10_nadie IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea8_nadie, 0)
        + COALESCE(tarea9_nadie, 0)
        + COALESCE(tarea10_nadie, 0)
    END AS n_tareas_nadie,

    -- SPSS COUNT de categorías válidas.
    (
      CAST(C3P302_1_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_2_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_3_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_4_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_5_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_6_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_7_gr3 IN (1, 2, 3) AS INT64)
    ) AS n_tareas_validas_1_7,

    (
      CAST(C3P302_8_gr4 IN (1, 2, 4) AS INT64)
      + CAST(C3P302_9_gr4 IN (1, 2, 4) AS INT64)
      + CAST(C3P302_10_gr4 IN (1, 2, 4) AS INT64)
    ) AS n_tareas_validas_8_10

  FROM binary_tasks
),

total_valid_tasks AS (
  SELECT
    *,
    n_tareas_validas_1_7
      + n_tareas_validas_8_10
      AS n_tareas_validas_p302
  FROM task_counts
),

task_proportions AS (
  SELECT
    *,

    SAFE_DIVIDE(
      n_tareas_femeninas,
      NULLIF(n_tareas_validas_p302, 0)
    ) AS prop_tareas_femeninas,

    SAFE_DIVIDE(
      n_tareas_masculinas,
      NULLIF(n_tareas_validas_p302, 0)
    ) AS prop_tareas_masculinas,

    SAFE_DIVIDE(
      n_tareas_nna,
      NULLIF(n_tareas_validas_1_7, 0)
    ) AS prop_tareas_nna,

    SAFE_DIVIDE(
      n_tareas_nadie,
      NULLIF(n_tareas_validas_8_10, 0)
    ) AS prop_tareas_nadie

  FROM total_valid_tasks
),

final AS (
  SELECT
    *,

    CASE
      WHEN prop_tareas_femeninas IS NULL
        OR prop_tareas_masculinas IS NULL
      THEN NULL

      WHEN prop_tareas_femeninas
        > prop_tareas_masculinas
      THEN 1

      ELSE 0
    END AS predominio_femenino_tareas

  FROM task_proportions
)

SELECT *
FROM final
"""

# ------------------------------------------------------------
# 5. Guardar y ejecutar
# ------------------------------------------------------------

(SQL_DIR / "stage3_syntax_31_roles_genero.sql").write_text(
    sql_31_roles,
    encoding="utf-8"
)

print(sql_31_roles)

client.query(sql_31_roles).result()

print(
    "Bloque SPSS 3.1 de roles de género "
    "y tareas del hogar creado correctamente."
)


# ------------------------------------------------------------
# 6. Validar resultados
# ------------------------------------------------------------

validation_roles = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    predominio_femenino_tareas IS NOT NULL
    AND predominio_femenino_tareas NOT IN (0, 1)
  ) AS invalid_predominio,

  COUNTIF(
    n_tareas_femeninas IS NOT NULL
    AND n_tareas_femeninas NOT BETWEEN 0 AND 10
  ) AS invalid_femeninas,

  COUNTIF(
    n_tareas_masculinas IS NOT NULL
    AND n_tareas_masculinas NOT BETWEEN 0 AND 10
  ) AS invalid_masculinas,

  COUNTIF(
    n_tareas_nna IS NOT NULL
    AND n_tareas_nna NOT BETWEEN 0 AND 7
  ) AS invalid_nna,

  COUNTIF(
    n_tareas_nadie IS NOT NULL
    AND n_tareas_nadie NOT BETWEEN 0 AND 3
  ) AS invalid_nadie,

  COUNTIF(
    n_tareas_validas_p302 NOT BETWEEN 0 AND 10
  ) AS invalid_validas,

  COUNTIF(
    prop_tareas_femeninas IS NOT NULL
    AND prop_tareas_femeninas NOT BETWEEN 0 AND 1
  ) AS invalid_prop_fem,

  COUNTIF(
    prop_tareas_masculinas IS NOT NULL
    AND prop_tareas_masculinas NOT BETWEEN 0 AND 1
  ) AS invalid_prop_masc,

  COUNTIF(
    prop_tareas_nna IS NOT NULL
    AND prop_tareas_nna NOT BETWEEN 0 AND 1
  ) AS invalid_prop_nna,

  COUNTIF(
    prop_tareas_nadie IS NOT NULL
    AND prop_tareas_nadie NOT BETWEEN 0 AND 1
  ) AS invalid_prop_nadie

FROM `{A}`
""").result().to_dataframe()

validation_roles.to_csv(
    LOG_DIR / "stage3_syntax_31_roles_validation.csv",
    index=False
)

display(validation_roles)

validation_columns = [
    column
    for column in validation_roles.columns
    if column.startswith("invalid_")
]

if (
    validation_roles.loc[0, validation_columns] > 0
).any():
    raise RuntimeError(
        "Falló la validación del bloque de roles de género."
    )


distribution_predominio = client.query(f"""
SELECT
  predominio_femenino_tareas,
  COUNT(*) AS n
FROM `{A}`
GROUP BY predominio_femenino_tareas
ORDER BY predominio_femenino_tareas
""").result().to_dataframe()

distribution_predominio.to_csv(
    LOG_DIR
    / "stage3_predominio_femenino_tareas_distribution.csv",
    index=False
)

display(distribution_predominio)

print(
    "Bloque de roles validado. "
    "predominio_femenino_tareas contiene 0, 1 o NULL."
)

Variables C3P302_1 a C3P302_10 verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

WITH source AS (
  SELECT
    *
  FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`
),

grouped_tasks AS (
  SELECT
    *,
    
        CASE
          WHEN C3P302_1 IN (2, 4, 6) THEN 1
          WHEN C3P302_1 IN (3, 5, 7) THEN 2
          WHEN C3P302_1 = 1 THEN 3
          ELSE NULL
        END AS C3P302_1_gr3
        ,

        CASE
          WHEN C3P302_2 IN (2, 4, 6) THEN 1
          WHEN C3P302_2 IN (3, 5, 7) THEN 2
          WHEN C3P302_2 = 1 THEN 3
          ELSE NULL
        END AS C3P302_2_gr3
        ,

        CASE
          WHEN C3P302_3 IN (2, 4, 6) THEN 1
          WHEN C3P302_3 IN (3, 5, 7) THEN 2
          WHEN C3P302_3 = 1 THEN 3
          ELSE NULL
        END AS C3P302_3_gr3
        ,

        CASE
          WHEN C3P302_4 IN (2, 4, 6) THEN 1
          WHEN C3P302_4 IN (3, 5, 7) THEN 2
    

,total_rows,invalid_predominio,invalid_femeninas,invalid_masculinas,invalid_nna,invalid_nadie,invalid_validas,invalid_prop_fem,invalid_prop_masc,invalid_prop_nna,invalid_prop_nadie
0,18807,0,0,0,0,0,0,0,0,0,0


,predominio_femenino_tareas,n
0,<NA>,6377
1,0,1319
2,1,11111


Bloque de roles validado. predominio_femenino_tareas contiene 0, 1 o NULL.


In [11]:
# ============================================================
# SPSS 3.1 — Roles de género y división del trabajo del hogar
# Fuente:
# 07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps
#
# Traduce:
# - C3P302_1..C3P302_7  -> grupos mujer/hombre/entrevistada-o
# - C3P302_8..C3P302_10 -> grupos mujer/hombre/nadie
# - Indicadores binarios por tarea
# - Conteos y proporciones
# - predominio_femenino_tareas
#
# Compatible con BigQuery Sandbox: CREATE OR REPLACE, sin UPDATE
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_roles = [
    f"C3P302_{i}"
    for i in range(1, 11)
]

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_roles = sorted(
    set(required_roles) - existing_columns
)

if missing_roles:
    raise RuntimeError(
        "No se puede crear el bloque de roles de género. "
        "Faltan variables fuente: "
        + ", ".join(missing_roles)
    )

print("Variables C3P302_1 a C3P302_10 verificadas.")


# ------------------------------------------------------------
# 2. Columnas derivadas que crea este bloque
# ------------------------------------------------------------

derived_roles = (
    [f"C3P302_{i}_gr3" for i in range(1, 8)]
    + [f"C3P302_{i}_gr4" for i in range(8, 11)]
    + [f"tarea{i}_fem" for i in range(1, 11)]
    + [f"tarea{i}_masc" for i in range(1, 11)]
    + [f"tarea{i}_nna" for i in range(1, 8)]
    + [f"tarea{i}_nadie" for i in range(8, 11)]
    + [
        "n_tareas_femeninas",
        "n_tareas_masculinas",
        "n_tareas_nna",
        "n_tareas_nadie",
        "n_tareas_validas_1_7",
        "n_tareas_validas_8_10",
        "n_tareas_validas_p302",
        "prop_tareas_femeninas",
        "prop_tareas_masculinas",
        "prop_tareas_nna",
        "prop_tareas_nadie",
        "predominio_femenino_tareas",
    ]
)

columns_to_replace = [
    column
    for column in derived_roles
    if column in existing_columns
]

if columns_to_replace:
    source_select = (
        "* EXCEPT("
        + ", ".join(
            f"`{column}`"
            for column in columns_to_replace
        )
        + ")"
    )
else:
    source_select = "*"


# ------------------------------------------------------------
# 3. Generar expresiones SQL repetitivas
# ------------------------------------------------------------

# P302.1 a P302.7:
# 2,4,6 -> mujer (1)
# 3,5,7 -> hombre (2)
# 1     -> entrevistada/o (3)

recodes_1_7 = []

for i in range(1, 8):
    recodes_1_7.append(
        f"""
        CASE
          WHEN C3P302_{i} IN (2, 4, 6) THEN 1
          WHEN C3P302_{i} IN (3, 5, 7) THEN 2
          WHEN C3P302_{i} = 1 THEN 3
          ELSE NULL
        END AS C3P302_{i}_gr3
        """
    )

# P302.8 a P302.10:
# 2,4,6 -> mujer (1)
# 3,5,7 -> hombre (2)
# 8     -> nadie (4)

recodes_8_10 = []

for i in range(8, 11):
    recodes_8_10.append(
        f"""
        CASE
          WHEN C3P302_{i} IN (2, 4, 6) THEN 1
          WHEN C3P302_{i} IN (3, 5, 7) THEN 2
          WHEN C3P302_{i} = 8 THEN 4
          ELSE NULL
        END AS C3P302_{i}_gr4
        """
    )

recode_expressions = ",\n".join(
    recodes_1_7 + recodes_8_10
)


# Indicadores binarios para tareas 1 a 7.
# Cuando la variable agrupada es missing, SPSS produce SYSMIS.

binary_1_7 = []

for i in range(1, 8):
    binary_1_7.extend([
        f"""
        CASE
          WHEN C3P302_{i}_gr3 IS NULL THEN NULL
          WHEN C3P302_{i}_gr3 = 1 THEN 1
          ELSE 0
        END AS tarea{i}_fem
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr3 IS NULL THEN NULL
          WHEN C3P302_{i}_gr3 = 2 THEN 1
          ELSE 0
        END AS tarea{i}_masc
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr3 IS NULL THEN NULL
          WHEN C3P302_{i}_gr3 = 3 THEN 1
          ELSE 0
        END AS tarea{i}_nna
        """,
    ])

# Indicadores binarios para tareas 8 a 10.

binary_8_10 = []

for i in range(8, 11):
    binary_8_10.extend([
        f"""
        CASE
          WHEN C3P302_{i}_gr4 IS NULL THEN NULL
          WHEN C3P302_{i}_gr4 = 1 THEN 1
          ELSE 0
        END AS tarea{i}_fem
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr4 IS NULL THEN NULL
          WHEN C3P302_{i}_gr4 = 2 THEN 1
          ELSE 0
        END AS tarea{i}_masc
        """,
        f"""
        CASE
          WHEN C3P302_{i}_gr4 IS NULL THEN NULL
          WHEN C3P302_{i}_gr4 = 4 THEN 1
          ELSE 0
        END AS tarea{i}_nadie
        """,
    ])

binary_expressions = ",\n".join(
    binary_1_7 + binary_8_10
)


# ------------------------------------------------------------
# 4. SQL principal
# ------------------------------------------------------------

sql_31_roles = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH source AS (
  SELECT
    {source_select}
  FROM `{A}`
),

grouped_tasks AS (
  SELECT
    *,
    {recode_expressions}
  FROM source
),

binary_tasks AS (
  SELECT
    *,
    {binary_expressions}
  FROM grouped_tasks
),

task_counts AS (
  SELECT
    *,

    -- SPSS SUM ignora componentes missing.
    -- Si todos están missing, el resultado queda NULL.

    CASE
      WHEN
        tarea1_fem IS NULL
        AND tarea2_fem IS NULL
        AND tarea3_fem IS NULL
        AND tarea4_fem IS NULL
        AND tarea5_fem IS NULL
        AND tarea6_fem IS NULL
        AND tarea7_fem IS NULL
        AND tarea8_fem IS NULL
        AND tarea9_fem IS NULL
        AND tarea10_fem IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea1_fem, 0)
        + COALESCE(tarea2_fem, 0)
        + COALESCE(tarea3_fem, 0)
        + COALESCE(tarea4_fem, 0)
        + COALESCE(tarea5_fem, 0)
        + COALESCE(tarea6_fem, 0)
        + COALESCE(tarea7_fem, 0)
        + COALESCE(tarea8_fem, 0)
        + COALESCE(tarea9_fem, 0)
        + COALESCE(tarea10_fem, 0)
    END AS n_tareas_femeninas,

    CASE
      WHEN
        tarea1_masc IS NULL
        AND tarea2_masc IS NULL
        AND tarea3_masc IS NULL
        AND tarea4_masc IS NULL
        AND tarea5_masc IS NULL
        AND tarea6_masc IS NULL
        AND tarea7_masc IS NULL
        AND tarea8_masc IS NULL
        AND tarea9_masc IS NULL
        AND tarea10_masc IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea1_masc, 0)
        + COALESCE(tarea2_masc, 0)
        + COALESCE(tarea3_masc, 0)
        + COALESCE(tarea4_masc, 0)
        + COALESCE(tarea5_masc, 0)
        + COALESCE(tarea6_masc, 0)
        + COALESCE(tarea7_masc, 0)
        + COALESCE(tarea8_masc, 0)
        + COALESCE(tarea9_masc, 0)
        + COALESCE(tarea10_masc, 0)
    END AS n_tareas_masculinas,

    CASE
      WHEN
        tarea1_nna IS NULL
        AND tarea2_nna IS NULL
        AND tarea3_nna IS NULL
        AND tarea4_nna IS NULL
        AND tarea5_nna IS NULL
        AND tarea6_nna IS NULL
        AND tarea7_nna IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea1_nna, 0)
        + COALESCE(tarea2_nna, 0)
        + COALESCE(tarea3_nna, 0)
        + COALESCE(tarea4_nna, 0)
        + COALESCE(tarea5_nna, 0)
        + COALESCE(tarea6_nna, 0)
        + COALESCE(tarea7_nna, 0)
    END AS n_tareas_nna,

    CASE
      WHEN
        tarea8_nadie IS NULL
        AND tarea9_nadie IS NULL
        AND tarea10_nadie IS NULL
      THEN NULL
      ELSE
        COALESCE(tarea8_nadie, 0)
        + COALESCE(tarea9_nadie, 0)
        + COALESCE(tarea10_nadie, 0)
    END AS n_tareas_nadie,

    -- SPSS COUNT de categorías válidas.
    (
      CAST(C3P302_1_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_2_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_3_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_4_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_5_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_6_gr3 IN (1, 2, 3) AS INT64)
      + CAST(C3P302_7_gr3 IN (1, 2, 3) AS INT64)
    ) AS n_tareas_validas_1_7,

    (
      CAST(C3P302_8_gr4 IN (1, 2, 4) AS INT64)
      + CAST(C3P302_9_gr4 IN (1, 2, 4) AS INT64)
      + CAST(C3P302_10_gr4 IN (1, 2, 4) AS INT64)
    ) AS n_tareas_validas_8_10

  FROM binary_tasks
),

total_valid_tasks AS (
  SELECT
    *,
    n_tareas_validas_1_7
      + n_tareas_validas_8_10
      AS n_tareas_validas_p302
  FROM task_counts
),

task_proportions AS (
  SELECT
    *,

    SAFE_DIVIDE(
      n_tareas_femeninas,
      NULLIF(n_tareas_validas_p302, 0)
    ) AS prop_tareas_femeninas,

    SAFE_DIVIDE(
      n_tareas_masculinas,
      NULLIF(n_tareas_validas_p302, 0)
    ) AS prop_tareas_masculinas,

    SAFE_DIVIDE(
      n_tareas_nna,
      NULLIF(n_tareas_validas_1_7, 0)
    ) AS prop_tareas_nna,

    SAFE_DIVIDE(
      n_tareas_nadie,
      NULLIF(n_tareas_validas_8_10, 0)
    ) AS prop_tareas_nadie

  FROM total_valid_tasks
),

final AS (
  SELECT
    *,

    CASE
      WHEN prop_tareas_femeninas IS NULL
        OR prop_tareas_masculinas IS NULL
      THEN NULL

      WHEN prop_tareas_femeninas
        > prop_tareas_masculinas
      THEN 1

      ELSE 0
    END AS predominio_femenino_tareas

  FROM task_proportions
)

SELECT *
FROM final
"""

# ------------------------------------------------------------
# 5. Guardar y ejecutar
# ------------------------------------------------------------

(SQL_DIR / "stage3_syntax_31_roles_genero.sql").write_text(
    sql_31_roles,
    encoding="utf-8"
)

print(sql_31_roles)

client.query(sql_31_roles).result()

print(
    "Bloque SPSS 3.1 de roles de género "
    "y tareas del hogar creado correctamente."
)


# ------------------------------------------------------------
# 6. Validar resultados
# ------------------------------------------------------------

validation_roles = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    predominio_femenino_tareas IS NOT NULL
    AND predominio_femenino_tareas NOT IN (0, 1)
  ) AS invalid_predominio,

  COUNTIF(
    n_tareas_femeninas IS NOT NULL
    AND n_tareas_femeninas NOT BETWEEN 0 AND 10
  ) AS invalid_femeninas,

  COUNTIF(
    n_tareas_masculinas IS NOT NULL
    AND n_tareas_masculinas NOT BETWEEN 0 AND 10
  ) AS invalid_masculinas,

  COUNTIF(
    n_tareas_nna IS NOT NULL
    AND n_tareas_nna NOT BETWEEN 0 AND 7
  ) AS invalid_nna,

  COUNTIF(
    n_tareas_nadie IS NOT NULL
    AND n_tareas_nadie NOT BETWEEN 0 AND 3
  ) AS invalid_nadie,

  COUNTIF(
    n_tareas_validas_p302 NOT BETWEEN 0 AND 10
  ) AS invalid_validas,

  COUNTIF(
    prop_tareas_femeninas IS NOT NULL
    AND prop_tareas_femeninas NOT BETWEEN 0 AND 1
  ) AS invalid_prop_fem,

  COUNTIF(
    prop_tareas_masculinas IS NOT NULL
    AND prop_tareas_masculinas NOT BETWEEN 0 AND 1
  ) AS invalid_prop_masc,

  COUNTIF(
    prop_tareas_nna IS NOT NULL
    AND prop_tareas_nna NOT BETWEEN 0 AND 1
  ) AS invalid_prop_nna,

  COUNTIF(
    prop_tareas_nadie IS NOT NULL
    AND prop_tareas_nadie NOT BETWEEN 0 AND 1
  ) AS invalid_prop_nadie

FROM `{A}`
""").result().to_dataframe()

validation_roles.to_csv(
    LOG_DIR / "stage3_syntax_31_roles_validation.csv",
    index=False
)

display(validation_roles)

validation_columns = [
    column
    for column in validation_roles.columns
    if column.startswith("invalid_")
]

if (
    validation_roles.loc[0, validation_columns] > 0
).any():
    raise RuntimeError(
        "Falló la validación del bloque de roles de género."
    )


distribution_predominio = client.query(f"""
SELECT
  predominio_femenino_tareas,
  COUNT(*) AS n
FROM `{A}`
GROUP BY predominio_femenino_tareas
ORDER BY predominio_femenino_tareas
""").result().to_dataframe()

distribution_predominio.to_csv(
    LOG_DIR
    / "stage3_predominio_femenino_tareas_distribution.csv",
    index=False
)

display(distribution_predominio)

print(
    "Bloque de roles validado. "
    "predominio_femenino_tareas contiene 0, 1 o NULL."
)

Variables C3P302_1 a C3P302_10 verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

WITH source AS (
  SELECT
    * EXCEPT(`C3P302_1_gr3`, `C3P302_2_gr3`, `C3P302_3_gr3`, `C3P302_4_gr3`, `C3P302_5_gr3`, `C3P302_6_gr3`, `C3P302_7_gr3`, `C3P302_8_gr4`, `C3P302_9_gr4`, `C3P302_10_gr4`, `tarea1_fem`, `tarea2_fem`, `tarea3_fem`, `tarea4_fem`, `tarea5_fem`, `tarea6_fem`, `tarea7_fem`, `tarea8_fem`, `tarea9_fem`, `tarea10_fem`, `tarea1_masc`, `tarea2_masc`, `tarea3_masc`, `tarea4_masc`, `tarea5_masc`, `tarea6_masc`, `tarea7_masc`, `tarea8_masc`, `tarea9_masc`, `tarea10_masc`, `tarea1_nna`, `tarea2_nna`, `tarea3_nna`, `tarea4_nna`, `tarea5_nna`, `tarea6_nna`, `tarea7_nna`, `tarea8_nadie`, `tarea9_nadie`, `tarea10_nadie`, `n_tareas_femeninas`, `n_tareas_masculinas`, `n_tareas_nna`, `n_tareas_nadie`, `n_tareas_validas_1_7`, `n_tareas_validas_8_10`, `n_tareas_validas_p302`, `prop_tareas_femeninas`, `prop_tareas_masculinas`, `prop_t

,total_rows,invalid_predominio,invalid_femeninas,invalid_masculinas,invalid_nna,invalid_nadie,invalid_validas,invalid_prop_fem,invalid_prop_masc,invalid_prop_nna,invalid_prop_nadie
0,18807,0,0,0,0,0,0,0,0,0,0


,predominio_femenino_tareas,n
0,<NA>,6377
1,0,1319
2,1,11111


Bloque de roles validado. predominio_femenino_tareas contiene 0, 1 o NULL.


In [12]:
# ============================================================
# SPSS 3.1 — Mitos sobre violencia sexual
# Fuente:
# 07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

existing = {
    f.name
    for f in client.get_table(A).schema
}

required = [
    "C3P303_1",
    "C3P303_3",
    "C3P303_4",
    "C3P303_5",
]

missing = sorted(set(required) - existing)

if missing:
    raise RuntimeError(
        "Faltan variables fuente: "
        + ", ".join(missing)
    )


replace = [
    "mito_locas",
    "mito_pobreza",
    "mito_fuera_casa",
    "mito_sitios_oscuros",
    "n_mitos",
    "cree_al_menos_un_mito",
    "n_mitos_cat",
]

drop = [c for c in replace if c in existing]

if drop:
    select_sql = (
        "* EXCEPT("
        + ",".join(drop)
        + ")"
    )
else:
    select_sql = "*"


sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

WITH base AS (

SELECT
{select_sql}
FROM `{A}`

),

recodes AS (

SELECT
*,

CASE
WHEN C3P303_1=1 THEN 1
WHEN C3P303_1=2 THEN 0
ELSE NULL
END AS mito_locas,

CASE
WHEN C3P303_3=1 THEN 1
WHEN C3P303_3=2 THEN 0
ELSE NULL
END AS mito_pobreza,

CASE
WHEN C3P303_4=1 THEN 1
WHEN C3P303_4=2 THEN 0
ELSE NULL
END AS mito_fuera_casa,

CASE
WHEN C3P303_5=1 THEN 1
WHEN C3P303_5=2 THEN 0
ELSE NULL
END AS mito_sitios_oscuros

FROM base

),

indicators AS (

SELECT
*,

CASE
WHEN mito_locas IS NULL
AND mito_pobreza IS NULL
AND mito_fuera_casa IS NULL
AND mito_sitios_oscuros IS NULL
THEN NULL

ELSE
COALESCE(mito_locas,0)
+
COALESCE(mito_pobreza,0)
+
COALESCE(mito_fuera_casa,0)
+
COALESCE(mito_sitios_oscuros,0)

END AS n_mitos

FROM recodes

),

final AS (

SELECT
*,

CASE
WHEN n_mitos IS NULL
THEN NULL

WHEN n_mitos>=1
THEN 1

ELSE 0
END AS cree_al_menos_un_mito,

CASE
WHEN n_mitos IS NULL
THEN NULL

WHEN n_mitos=0
THEN 0

WHEN n_mitos=1
THEN 1

ELSE 2
END AS n_mitos_cat

FROM indicators

)

SELECT *
FROM final
"""

(SQL_DIR/"stage3_syntax_31_mitos.sql").write_text(
    sql,
    encoding="utf-8"
)

client.query(sql).result()

print("Bloque SPSS 3.1 - Mitos creado.")

Bloque SPSS 3.1 - Mitos creado.


In [13]:
validation = client.query(f"""
SELECT

COUNTIF(
mito_locas NOT IN (0,1)
AND mito_locas IS NOT NULL
) bad_locas,

COUNTIF(
mito_pobreza NOT IN (0,1)
AND mito_pobreza IS NOT NULL
) bad_pobreza,

COUNTIF(
mito_fuera_casa NOT IN (0,1)
AND mito_fuera_casa IS NOT NULL
) bad_fuera,

COUNTIF(
mito_sitios_oscuros NOT IN (0,1)
AND mito_sitios_oscuros IS NOT NULL
) bad_oscuros,

COUNTIF(
cree_al_menos_un_mito NOT IN (0,1)
AND cree_al_menos_un_mito IS NOT NULL
) bad_indicator,

COUNTIF(
n_mitos NOT BETWEEN 0 AND 4
AND n_mitos IS NOT NULL
) bad_count,

COUNTIF(
n_mitos_cat NOT IN (0,1,2)
AND n_mitos_cat IS NOT NULL
) bad_cat

FROM `{A}`
""").result().to_dataframe()

display(validation)

if (validation.iloc[0] > 0).any():
    raise RuntimeError(
        "Falló la validación del bloque de mitos."
    )

print("Bloque de mitos validado correctamente.")

,bad_locas,bad_pobreza,bad_fuera,bad_oscuros,bad_indicator,bad_count,bad_cat
0,0,0,0,0,0,0,0


Bloque de mitos validado correctamente.


In [14]:
# ============================================================
# Traducción SPSS 3.2 — VP_HOGAR
# Fuente:
# 08_CRS04_3.2 Violencia en el hogar_ver6.sps
# Compatible con BigQuery Sandbox: CREATE OR REPLACE, sin UPDATE
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_vp_hogar = ["C3P203"]

for i in range(1, 12):
    required_vp_hogar.extend([
        f"C3P201_{i}",
        f"C3P201A_{i}",
        f"C3P201C_{i}",
        f"C3P201D_{i}",
        f"C3P201E_{i}",
        f"C3P201F_{i}",
    ])

analytical_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_vp_hogar = sorted(
    set(required_vp_hogar) - analytical_columns
)

if missing_vp_hogar:
    raise RuntimeError(
        "No se puede crear VP_HOGAR. "
        "Faltan estas variables en analytical: "
        + ", ".join(missing_vp_hogar)
    )

print("Variables fuente de VP_HOGAR verificadas.")


# ------------------------------------------------------------
# 2. Construir condición de los 11 ítems
# ------------------------------------------------------------

valid_aggressor_codes = "1, 2, 3, 4, 19"

vp_item_conditions = []

for i in range(1, 12):
    item_condition = f"""
    (
      `C3P201_{i}` = 1
      AND (
        `C3P201A_{i}` IN ({valid_aggressor_codes})

        OR `C3P201E_{i}` IN ({valid_aggressor_codes})

        OR (
          `C3P201A_{i}` NOT IN ({valid_aggressor_codes})
          AND `C3P201C_{i}` = 1
          AND `C3P201D_{i}` = 1
        )

        OR (
          `C3P201E_{i}` NOT IN ({valid_aggressor_codes})
          AND `C3P201F_{i}` = 1
        )
      )
    )
    """

    vp_item_conditions.append(item_condition)

any_vp_item = "\nOR\n".join(vp_item_conditions)


# ------------------------------------------------------------
# 3. Preparar reejecución segura
# ------------------------------------------------------------

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

select_prefix = (
    "* EXCEPT(VP_HOGAR)"
    if "VP_HOGAR" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 4. Crear o reemplazar VP_HOGAR
# ------------------------------------------------------------

vp_hogar_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN `C3P203` = 1
     AND (
       {any_vp_item}
     )
    THEN 1

    ELSE 0
  END AS VP_HOGAR

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vp_hogar.sql").write_text(
    vp_hogar_sql,
    encoding="utf-8"
)

print(vp_hogar_sql)

client.query(vp_hogar_sql).result()

print("VP_HOGAR creado correctamente.")


# ------------------------------------------------------------
# 5. Validar dominio y distribución
# ------------------------------------------------------------

vp_hogar_distribution = client.query(f"""
SELECT
  VP_HOGAR,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VP_HOGAR
ORDER BY VP_HOGAR
""").result().to_dataframe()

vp_hogar_distribution.to_csv(
    LOG_DIR / "stage3_vp_hogar_distribution.csv",
    index=False
)

display(vp_hogar_distribution)


invalid_vp_hogar = client.query(f"""
SELECT
  COUNTIF(
    VP_HOGAR IS NULL
    OR VP_HOGAR NOT IN (0, 1)
  ) AS invalid_values
FROM `{A}`
""").result().to_dataframe()

display(invalid_vp_hogar)

if invalid_vp_hogar.iloc[0]["invalid_values"] > 0:
    raise RuntimeError(
        "VP_HOGAR contiene valores distintos de 0 o 1."
    )

print("VP_HOGAR validado: solo contiene 0 y 1.")

Variables fuente de VP_HOGAR verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN `C3P203` = 1
     AND (
       
    (
      `C3P201_1` = 1
      AND (
        `C3P201A_1` IN (1, 2, 3, 4, 19)

        OR `C3P201E_1` IN (1, 2, 3, 4, 19)

        OR (
          `C3P201A_1` NOT IN (1, 2, 3, 4, 19)
          AND `C3P201C_1` = 1
          AND `C3P201D_1` = 1
        )

        OR (
          `C3P201E_1` NOT IN (1, 2, 3, 4, 19)
          AND `C3P201F_1` = 1
        )
      )
    )
    
OR

    (
      `C3P201_2` = 1
      AND (
        `C3P201A_2` IN (1, 2, 3, 4, 19)

        OR `C3P201E_2` IN (1, 2, 3, 4, 19)

        OR (
          `C3P201A_2` NOT IN (1, 2, 3, 4, 19)
          AND `C3P201C_2` = 1
          AND `C3P201D_2` = 1
        )

        OR (
          `C3P201E_2` NOT IN (1, 2, 3, 4, 19)
          AND `C3P201F_2` = 1
        )
      )
    )
    
OR

    (
      `C3P201_3` = 1
      AND (


,VP_HOGAR,n
0,0,13538
1,1,5269


,invalid_values
0,0


VP_HOGAR validado: solo contiene 0 y 1.


In [15]:
# ============================================================
# Traducción SPSS 3.2 — VF_HOGAR
# Fuente:
# 08_CRS04_3.2 Violencia en el hogar_ver6.sps
#
# Definición:
# Violencia física ejercida por madre, padre o quien haga
# sus veces durante los últimos 12 meses.
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE ni ALTER TABLE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_vf_hogar = ["C3P207"]

for i in range(1, 8):
    required_vf_hogar.extend([
        f"C3P205_{i}",
        f"C3P205A_{i}",
        f"C3P205C_{i}",
        f"C3P205D_{i}",
        f"C3P205E_{i}",
        f"C3P205F_{i}",
    ])

analytical_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_vf_hogar = sorted(
    set(required_vf_hogar) - analytical_columns
)

if missing_vf_hogar:
    raise RuntimeError(
        "No se puede crear VF_HOGAR. "
        "Faltan estas variables en analytical: "
        + ", ".join(missing_vf_hogar)
    )

print("Variables fuente de VF_HOGAR verificadas.")


# ------------------------------------------------------------
# 2. Construir condición de los 7 ítems
# ------------------------------------------------------------

valid_aggressor_codes = "1, 2, 3, 4, 19"

vf_item_conditions = []

for i in range(1, 8):
    item_condition = f"""
    (
      `C3P205_{i}` = 1
      AND (
        `C3P205A_{i}` IN ({valid_aggressor_codes})

        OR `C3P205E_{i}` IN ({valid_aggressor_codes})

        OR (
          `C3P205A_{i}` NOT IN ({valid_aggressor_codes})
          AND `C3P205C_{i}` = 1
          AND `C3P205D_{i}` = 1
        )

        OR (
          `C3P205E_{i}` NOT IN ({valid_aggressor_codes})
          AND `C3P205F_{i}` = 1
        )
      )
    )
    """

    vf_item_conditions.append(item_condition)

any_vf_item = "\nOR\n".join(vf_item_conditions)


# ------------------------------------------------------------
# 3. Preparar reejecución segura
# ------------------------------------------------------------

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

select_prefix = (
    "* EXCEPT(VF_HOGAR)"
    if "VF_HOGAR" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 4. Crear o reemplazar VF_HOGAR
# ------------------------------------------------------------

vf_hogar_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN `C3P207` = 1
     AND (
       {any_vf_item}
     )
    THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vf_hogar.sql").write_text(
    vf_hogar_sql,
    encoding="utf-8"
)

print(vf_hogar_sql)

client.query(vf_hogar_sql).result()

print("VF_HOGAR creado correctamente.")


# ------------------------------------------------------------
# 5. Validar dominio y distribución
# ------------------------------------------------------------

vf_hogar_distribution = client.query(f"""
SELECT
  VF_HOGAR,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VF_HOGAR
ORDER BY VF_HOGAR
""").result().to_dataframe()

vf_hogar_distribution.to_csv(
    LOG_DIR / "stage3_vf_hogar_distribution.csv",
    index=False
)

display(vf_hogar_distribution)


invalid_vf_hogar = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VF_HOGAR IS NULL
    OR VF_HOGAR NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VF_HOGAR = 0) AS zero_values,

  COUNTIF(VF_HOGAR = 1) AS one_values

FROM `{A}`
""").result().to_dataframe()

invalid_vf_hogar.to_csv(
    LOG_DIR / "stage3_vf_hogar_validation.csv",
    index=False
)

display(invalid_vf_hogar)

if invalid_vf_hogar.iloc[0]["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VF_HOGAR alteró el universo analítico."
    )

if invalid_vf_hogar.iloc[0]["invalid_values"] > 0:
    raise RuntimeError(
        "VF_HOGAR contiene valores distintos de 0 o 1."
    )

print(
    "VF_HOGAR validado: 18,807 filas "
    "y valores exclusivamente 0 o 1."
)

Variables fuente de VF_HOGAR verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN `C3P207` = 1
     AND (
       
    (
      `C3P205_1` = 1
      AND (
        `C3P205A_1` IN (1, 2, 3, 4, 19)

        OR `C3P205E_1` IN (1, 2, 3, 4, 19)

        OR (
          `C3P205A_1` NOT IN (1, 2, 3, 4, 19)
          AND `C3P205C_1` = 1
          AND `C3P205D_1` = 1
        )

        OR (
          `C3P205E_1` NOT IN (1, 2, 3, 4, 19)
          AND `C3P205F_1` = 1
        )
      )
    )
    
OR

    (
      `C3P205_2` = 1
      AND (
        `C3P205A_2` IN (1, 2, 3, 4, 19)

        OR `C3P205E_2` IN (1, 2, 3, 4, 19)

        OR (
          `C3P205A_2` NOT IN (1, 2, 3, 4, 19)
          AND `C3P205C_2` = 1
          AND `C3P205D_2` = 1
        )

        OR (
          `C3P205E_2` NOT IN (1, 2, 3, 4, 19)
          AND `C3P205F_2` = 1
        )
      )
    )
    
OR

    (
      `C3P205_3` = 1
      AND (


,VF_HOGAR,n
0,0,15673
1,1,3134


,total_rows,invalid_values,zero_values,one_values
0,18807,0,15673,3134


VF_HOGAR validado: 18,807 filas y valores exclusivamente 0 o 1.


In [16]:
# ============================================================
# Traducción SPSS 3.2 — VF_HOGAR_01
# Privación de necesidades básicas
#
# Fuente:
# 08_CRS04_3.2 Violencia en el hogar_ver6.sps
#
# SPSS:
# IF (C3P121 = 1) VF_HOGAR_01 = 1.
# RECODE VF_HOGAR_01 (SYSMIS = 0).
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variable fuente
# ------------------------------------------------------------

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

if "C3P121" not in existing_columns:
    raise RuntimeError(
        "No se puede crear VF_HOGAR_01: "
        "la variable C3P121 no existe en analytical."
    )

print("Variable fuente C3P121 verificada.")


# ------------------------------------------------------------
# 2. Preparar reejecución segura
# ------------------------------------------------------------

select_prefix = (
    "* EXCEPT(VF_HOGAR_01)"
    if "VF_HOGAR_01" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 3. Crear o reemplazar VF_HOGAR_01
# ------------------------------------------------------------

vf_hogar_01_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN C3P121 = 1 THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR_01 (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR_01

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vf_hogar_01.sql").write_text(
    vf_hogar_01_sql,
    encoding="utf-8"
)

print(vf_hogar_01_sql)

client.query(vf_hogar_01_sql).result()

print("VF_HOGAR_01 creado correctamente.")


# ------------------------------------------------------------
# 4. Validar dominio, universo y consistencia
# ------------------------------------------------------------

vf_hogar_01_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VF_HOGAR_01 IS NULL
    OR VF_HOGAR_01 NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VF_HOGAR_01 = 0) AS zero_values,

  COUNTIF(VF_HOGAR_01 = 1) AS one_values,

  COUNTIF(
    C3P121 = 1
    AND VF_HOGAR_01 != 1
  ) AS source_yes_not_indicator,

  COUNTIF(
    (C3P121 != 1 OR C3P121 IS NULL)
    AND VF_HOGAR_01 != 0
  ) AS source_not_yes_but_indicator

FROM `{A}`
""").result().to_dataframe()

vf_hogar_01_validation.to_csv(
    LOG_DIR / "stage3_vf_hogar_01_validation.csv",
    index=False
)

display(vf_hogar_01_validation)

r = vf_hogar_01_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VF_HOGAR_01 alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VF_HOGAR_01 contiene valores distintos de 0 o 1."
    )

if (
    r["source_yes_not_indicator"] > 0
    or r["source_not_yes_but_indicator"] > 0
):
    raise RuntimeError(
        "VF_HOGAR_01 no coincide con la regla C3P121 = 1."
    )

print(
    "VF_HOGAR_01 validado: "
    "18,807 filas, dominio 0/1 y correspondencia exacta con C3P121."
)


# ------------------------------------------------------------
# 5. Mostrar distribución
# ------------------------------------------------------------

vf_hogar_01_distribution = client.query(f"""
SELECT
  VF_HOGAR_01,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VF_HOGAR_01
ORDER BY VF_HOGAR_01
""").result().to_dataframe()

vf_hogar_01_distribution.to_csv(
    LOG_DIR / "stage3_vf_hogar_01_distribution.csv",
    index=False
)

display(vf_hogar_01_distribution)

Variable fuente C3P121 verificada.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN C3P121 = 1 THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR_01 (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR_01

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VF_HOGAR_01 creado correctamente.


,total_rows,invalid_values,zero_values,one_values,source_yes_not_indicator,source_not_yes_but_indicator
0,18807,0,18484,323,0,0


VF_HOGAR_01 validado: 18,807 filas, dominio 0/1 y correspondencia exacta con C3P121.


,VF_HOGAR_01,n
0,0,18484
1,1,323


In [17]:
# ============================================================
# Traducción SPSS 3.2 — VF_HOGAR_01
# Privación de necesidades básicas
#
# Fuente:
# 08_CRS04_3.2 Violencia en el hogar_ver6.sps
#
# SPSS:
# IF (C3P121 = 1) VF_HOGAR_01 = 1.
# RECODE VF_HOGAR_01 (SYSMIS = 0).
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variable fuente
# ------------------------------------------------------------

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

if "C3P121" not in existing_columns:
    raise RuntimeError(
        "No se puede crear VF_HOGAR_01: "
        "la variable C3P121 no existe en analytical."
    )

print("Variable fuente C3P121 verificada.")


# ------------------------------------------------------------
# 2. Preparar reejecución segura
# ------------------------------------------------------------

select_prefix = (
    "* EXCEPT(VF_HOGAR_01)"
    if "VF_HOGAR_01" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 3. Crear o reemplazar VF_HOGAR_01
# ------------------------------------------------------------

vf_hogar_01_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN C3P121 = 1 THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR_01 (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR_01

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vf_hogar_01.sql").write_text(
    vf_hogar_01_sql,
    encoding="utf-8"
)

print(vf_hogar_01_sql)

client.query(vf_hogar_01_sql).result()

print("VF_HOGAR_01 creado correctamente.")


# ------------------------------------------------------------
# 4. Validar dominio, universo y consistencia
# ------------------------------------------------------------

vf_hogar_01_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VF_HOGAR_01 IS NULL
    OR VF_HOGAR_01 NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VF_HOGAR_01 = 0) AS zero_values,

  COUNTIF(VF_HOGAR_01 = 1) AS one_values,

  COUNTIF(
    C3P121 = 1
    AND VF_HOGAR_01 != 1
  ) AS source_yes_not_indicator,

  COUNTIF(
    (C3P121 != 1 OR C3P121 IS NULL)
    AND VF_HOGAR_01 != 0
  ) AS source_not_yes_but_indicator

FROM `{A}`
""").result().to_dataframe()

vf_hogar_01_validation.to_csv(
    LOG_DIR / "stage3_vf_hogar_01_validation.csv",
    index=False
)

display(vf_hogar_01_validation)

r = vf_hogar_01_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VF_HOGAR_01 alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VF_HOGAR_01 contiene valores distintos de 0 o 1."
    )

if (
    r["source_yes_not_indicator"] > 0
    or r["source_not_yes_but_indicator"] > 0
):
    raise RuntimeError(
        "VF_HOGAR_01 no coincide con la regla C3P121 = 1."
    )

print(
    "VF_HOGAR_01 validado: "
    "18,807 filas, dominio 0/1 y correspondencia exacta con C3P121."
)


# ------------------------------------------------------------
# 5. Mostrar distribución
# ------------------------------------------------------------

vf_hogar_01_distribution = client.query(f"""
SELECT
  VF_HOGAR_01,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VF_HOGAR_01
ORDER BY VF_HOGAR_01
""").result().to_dataframe()

vf_hogar_01_distribution.to_csv(
    LOG_DIR / "stage3_vf_hogar_01_distribution.csv",
    index=False
)

display(vf_hogar_01_distribution)

Variable fuente C3P121 verificada.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  * EXCEPT(VF_HOGAR_01),

  CASE
    WHEN C3P121 = 1 THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR_01 (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR_01

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VF_HOGAR_01 creado correctamente.


,total_rows,invalid_values,zero_values,one_values,source_yes_not_indicator,source_not_yes_but_indicator
0,18807,0,18484,323,0,0


VF_HOGAR_01 validado: 18,807 filas, dominio 0/1 y correspondencia exacta con C3P121.


,VF_HOGAR_01,n
0,0,18484
1,1,323


In [18]:
# ============================================================
# Traducción SPSS 3.2 — VF_HOGAR_03
# Desprotección familiar en los últimos 12 meses
#
# Fuente:
# 08_CRS04_3.2 Violencia en el hogar_ver6.sps
#
# SPSS:
# IF (
#   (C3P216A_1 = 1 & C3P216A_1C = 1) |
#   ...
#   (C3P216C_5 = 1 & C3P216C_5C = 1)
# ) VF_HOGAR_03 = 1.
# RECODE VF_HOGAR_03 (SYSMIS = 0).
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_vf_hogar_03 = (
    [f"C3P216A_{i}" for i in range(1, 7)]
    + [f"C3P216A_{i}C" for i in range(1, 7)]
    + [f"C3P216C_{i}" for i in range(1, 6)]
    + [f"C3P216C_{i}C" for i in range(1, 6)]
)

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_vf_hogar_03 = sorted(
    set(required_vf_hogar_03) - existing_columns
)

if missing_vf_hogar_03:
    raise RuntimeError(
        "No se puede crear VF_HOGAR_03. "
        "Faltan estas variables en analytical: "
        + ", ".join(missing_vf_hogar_03)
    )

print("Variables fuente de VF_HOGAR_03 verificadas.")


# ------------------------------------------------------------
# 2. Construir las condiciones de desprotección
# ------------------------------------------------------------

conditions_a = [
    f"(`C3P216A_{i}` = 1 AND `C3P216A_{i}C` = 1)"
    for i in range(1, 7)
]

conditions_c = [
    f"(`C3P216C_{i}` = 1 AND `C3P216C_{i}C` = 1)"
    for i in range(1, 6)
]

any_vf_hogar_03 = "\n      OR ".join(
    conditions_a + conditions_c
)


# ------------------------------------------------------------
# 3. Preparar reejecución segura
# ------------------------------------------------------------

select_prefix = (
    "* EXCEPT(VF_HOGAR_03)"
    if "VF_HOGAR_03" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 4. Crear o reemplazar VF_HOGAR_03
# ------------------------------------------------------------

vf_hogar_03_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN (
      {any_vf_hogar_03}
    )
    THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR_03 (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR_03

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vf_hogar_03.sql").write_text(
    vf_hogar_03_sql,
    encoding="utf-8"
)

print(vf_hogar_03_sql)

client.query(vf_hogar_03_sql).result()

print("VF_HOGAR_03 creado correctamente.")


# ------------------------------------------------------------
# 5. Validar universo, dominio y correspondencia
# ------------------------------------------------------------

vf_hogar_03_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VF_HOGAR_03 IS NULL
    OR VF_HOGAR_03 NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VF_HOGAR_03 = 0) AS zero_values,

  COUNTIF(VF_HOGAR_03 = 1) AS one_values,

  COUNTIF(
    (
      {any_vf_hogar_03}
    )
    AND VF_HOGAR_03 != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    NOT (
      {any_vf_hogar_03}
    )
    AND VF_HOGAR_03 != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vf_hogar_03_validation.to_csv(
    LOG_DIR / "stage3_vf_hogar_03_validation.csv",
    index=False
)

display(vf_hogar_03_validation)

r = vf_hogar_03_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VF_HOGAR_03 alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VF_HOGAR_03 contiene valores distintos de 0 o 1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VF_HOGAR_03 no coincide con la lógica SPSS."
    )

print(
    "VF_HOGAR_03 validado: "
    "18,807 filas, dominio 0/1 y correspondencia exacta con SPSS."
)


# ------------------------------------------------------------
# 6. Mostrar distribución
# ------------------------------------------------------------

vf_hogar_03_distribution = client.query(f"""
SELECT
  VF_HOGAR_03,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VF_HOGAR_03
ORDER BY VF_HOGAR_03
""").result().to_dataframe()

vf_hogar_03_distribution.to_csv(
    LOG_DIR / "stage3_vf_hogar_03_distribution.csv",
    index=False
)

display(vf_hogar_03_distribution)

Variables fuente de VF_HOGAR_03 verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN (
      (`C3P216A_1` = 1 AND `C3P216A_1C` = 1)
      OR (`C3P216A_2` = 1 AND `C3P216A_2C` = 1)
      OR (`C3P216A_3` = 1 AND `C3P216A_3C` = 1)
      OR (`C3P216A_4` = 1 AND `C3P216A_4C` = 1)
      OR (`C3P216A_5` = 1 AND `C3P216A_5C` = 1)
      OR (`C3P216A_6` = 1 AND `C3P216A_6C` = 1)
      OR (`C3P216C_1` = 1 AND `C3P216C_1C` = 1)
      OR (`C3P216C_2` = 1 AND `C3P216C_2C` = 1)
      OR (`C3P216C_3` = 1 AND `C3P216C_3C` = 1)
      OR (`C3P216C_4` = 1 AND `C3P216C_4C` = 1)
      OR (`C3P216C_5` = 1 AND `C3P216C_5C` = 1)
    )
    THEN 1

    -- Equivale a:
    -- RECODE VF_HOGAR_03 (SYSMIS = 0).
    ELSE 0
  END AS VF_HOGAR_03

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VF_HOGAR_03 creado correctamente.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,15037,3770,0,0


VF_HOGAR_03 validado: 18,807 filas, dominio 0/1 y correspondencia exacta con SPSS.


,VF_HOGAR_03,n
0,0,15037
1,1,3770


In [19]:
# ============================================================
# SPSS 3.2 — VN_HOGAR1
# Negligencia en el hogar
#
# Logic:
# VN_HOGAR1 = 1 when VF_HOGAR_01 = 1 OR VF_HOGAR_03 = 1
# Otherwise 0.
#
# Compatible with BigQuery Sandbox:
# CREATE OR REPLACE TABLE, no UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# 1. Verify source variables
existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

required_vn_hogar1 = [
    "VF_HOGAR_01",
    "VF_HOGAR_03",
]

missing_vn_hogar1 = sorted(
    set(required_vn_hogar1) - existing_columns
)

if missing_vn_hogar1:
    raise RuntimeError(
        "Cannot create VN_HOGAR1. Missing source variables: "
        + ", ".join(missing_vn_hogar1)
    )

print("VN_HOGAR1 source variables verified.")


# 2. Safe rerun
select_prefix = (
    "* EXCEPT(VN_HOGAR1)"
    if "VN_HOGAR1" in existing_columns
    else "*"
)


# 3. Create or replace VN_HOGAR1
vn_hogar1_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN VF_HOGAR_01 = 1
      OR VF_HOGAR_03 = 1
    THEN 1

    ELSE 0
  END AS VN_HOGAR1

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vn_hogar1.sql").write_text(
    vn_hogar1_sql,
    encoding="utf-8"
)

print(vn_hogar1_sql)

client.query(vn_hogar1_sql).result()

print("VN_HOGAR1 created successfully.")


# 4. Validate universe, domain and consistency
vn_hogar1_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VN_HOGAR1 IS NULL
    OR VN_HOGAR1 NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VN_HOGAR1 = 0) AS zero_values,

  COUNTIF(VN_HOGAR1 = 1) AS one_values,

  COUNTIF(
    (VF_HOGAR_01 = 1 OR VF_HOGAR_03 = 1)
    AND VN_HOGAR1 != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    VF_HOGAR_01 != 1
    AND VF_HOGAR_03 != 1
    AND VN_HOGAR1 != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vn_hogar1_validation.to_csv(
    LOG_DIR / "stage3_vn_hogar1_validation.csv",
    index=False
)

display(vn_hogar1_validation)

r = vn_hogar1_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VN_HOGAR1 changed the analytical universe."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VN_HOGAR1 contains values outside 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VN_HOGAR1 does not match the expected source logic."
    )

print(
    "VN_HOGAR1 validated: "
    "18,807 rows, 0/1 domain, and exact consistency."
)


# 5. Distribution
vn_hogar1_distribution = client.query(f"""
SELECT
  VN_HOGAR1,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VN_HOGAR1
ORDER BY VN_HOGAR1
""").result().to_dataframe()

vn_hogar1_distribution.to_csv(
    LOG_DIR / "stage3_vn_hogar1_distribution.csv",
    index=False
)

display(vn_hogar1_distribution)

VN_HOGAR1 source variables verified.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN VF_HOGAR_01 = 1
      OR VF_HOGAR_03 = 1
    THEN 1

    ELSE 0
  END AS VN_HOGAR1

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VN_HOGAR1 created successfully.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,14842,3965,0,0


VN_HOGAR1 validated: 18,807 rows, 0/1 domain, and exact consistency.


,VN_HOGAR1,n
0,0,14842
1,1,3965


In [20]:
# ============================================================
# Traducción SPSS 3.2 — INDICADOR_8.3.5
# Violencia en el hogar (indicador consolidado)
#
# SPSS:
# =1 si:
#   VP_HOGAR=1
#   OR VF_HOGAR=1
#   OR VF_HOGAR_01=1
#   OR VF_HOGAR_03=1
#
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

existing = {
    f.name
    for f in client.get_table(A).schema
}

required = [
    "VP_HOGAR",
    "VF_HOGAR",
    "VF_HOGAR_01",
    "VF_HOGAR_03",
]

missing = sorted(set(required) - existing)

if missing:
    raise RuntimeError(
        "Faltan variables fuente: "
        + ", ".join(missing)
    )

select_prefix = (
    "* EXCEPT(INDICADOR_8_3_5)"
    if "INDICADOR_8_3_5" in existing
    else "*"
)

sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
    {select_prefix},

    CASE
        WHEN
            VP_HOGAR = 1
            OR VF_HOGAR = 1
            OR VF_HOGAR_01 = 1
            OR VF_HOGAR_03 = 1
        THEN 1
        ELSE 0
    END AS INDICADOR_8_3_5

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_indicador_836.sql").write_text(
    sql,
    encoding="utf-8"
)

print(sql)

client.query(sql).result()

print("INDICADOR_8_3_5 creado.")


CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
    *,

    CASE
        WHEN
            VP_HOGAR = 1
            OR VF_HOGAR = 1
            OR VF_HOGAR_01 = 1
            OR VF_HOGAR_03 = 1
        THEN 1
        ELSE 0
    END AS INDICADOR_8_3_5

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

INDICADOR_8_3_5 creado.


In [21]:
validation = client.query(f"""
SELECT
    COUNT(*) total_rows,

    COUNTIF(
        INDICADOR_8_3_5 NOT IN (0,1)
        AND INDICADOR_8_3_5 IS NOT NULL
    ) invalid_values,

    COUNTIF(
        (
            VP_HOGAR=1
            OR VF_HOGAR=1
            OR VF_HOGAR_01=1
            OR VF_HOGAR_03=1
        )
        AND INDICADOR_8_3_5!=1
    ) inconsistent_yes,

    COUNTIF(
        VP_HOGAR!=1
        AND VF_HOGAR!=1
        AND VF_HOGAR_01!=1
        AND VF_HOGAR_03!=1
        AND INDICADOR_8_3_5!=0
    ) inconsistent_no

FROM `{A}`
""").result().to_dataframe()

display(validation)

validation.to_csv(
    LOG_DIR / "stage3_indicador_836_validation.csv",
    index=False
)

r = validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError("Universo alterado.")

if r["invalid_values"] > 0:
    raise RuntimeError("Dominio inválido.")

if r["inconsistent_yes"] > 0 or r["inconsistent_no"] > 0:
    raise RuntimeError("La lógica no coincide con SPSS.")

print("INDICADOR_8_3_5 validado correctamente.")

,total_rows,invalid_values,inconsistent_yes,inconsistent_no
0,18807,0,0,0


INDICADOR_8_3_5 validado correctamente.


In [22]:
# ============================================================
# Traducción SPSS 3.2 — VP_o_VF_HOGAR
# Violencia psicológica y/o física en el hogar
#
# Fuente:
# 08_CRS04_3.2 Violencia en el hogar_ver6.sps
#
# SPSS:
# IF (VP_HOGAR = 1 | VF_HOGAR = 1) VP_o_VF_HOGAR = 1.
# RECODE VP_o_VF_HOGAR (SYSMIS = 0).
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

required_vp_o_vf_hogar = [
    "VP_HOGAR",
    "VF_HOGAR",
]

missing_vp_o_vf_hogar = sorted(
    set(required_vp_o_vf_hogar) - existing_columns
)

if missing_vp_o_vf_hogar:
    raise RuntimeError(
        "No se puede crear VP_o_VF_HOGAR. "
        "Faltan variables fuente: "
        + ", ".join(missing_vp_o_vf_hogar)
    )

print("Variables fuente de VP_o_VF_HOGAR verificadas.")


# ------------------------------------------------------------
# 2. Preparar reejecución segura
# ------------------------------------------------------------

select_prefix = (
    "* EXCEPT(VP_o_VF_HOGAR)"
    if "VP_o_VF_HOGAR" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 3. Crear o reemplazar VP_o_VF_HOGAR
# ------------------------------------------------------------

vp_o_vf_hogar_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN VP_HOGAR = 1
      OR VF_HOGAR = 1
    THEN 1

    -- Equivale a:
    -- RECODE VP_o_VF_HOGAR (SYSMIS = 0).
    ELSE 0
  END AS VP_o_VF_HOGAR

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vp_o_vf_hogar.sql").write_text(
    vp_o_vf_hogar_sql,
    encoding="utf-8"
)

print(vp_o_vf_hogar_sql)

client.query(vp_o_vf_hogar_sql).result()

print("VP_o_VF_HOGAR creado correctamente.")


# ------------------------------------------------------------
# 4. Validar universo, dominio y correspondencia
# ------------------------------------------------------------

vp_o_vf_hogar_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VP_o_VF_HOGAR IS NULL
    OR VP_o_VF_HOGAR NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VP_o_VF_HOGAR = 0) AS zero_values,

  COUNTIF(VP_o_VF_HOGAR = 1) AS one_values,

  COUNTIF(
    (VP_HOGAR = 1 OR VF_HOGAR = 1)
    AND VP_o_VF_HOGAR != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    VP_HOGAR != 1
    AND VF_HOGAR != 1
    AND VP_o_VF_HOGAR != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vp_o_vf_hogar_validation.to_csv(
    LOG_DIR / "stage3_vp_o_vf_hogar_validation.csv",
    index=False
)

display(vp_o_vf_hogar_validation)

r = vp_o_vf_hogar_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VP_o_VF_HOGAR alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VP_o_VF_HOGAR contiene valores fuera de 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VP_o_VF_HOGAR no coincide con la lógica SPSS."
    )

print(
    "VP_o_VF_HOGAR validado: "
    "18,807 filas, dominio 0/1 y correspondencia exacta."
)


# ------------------------------------------------------------
# 5. Mostrar distribución
# ------------------------------------------------------------

vp_o_vf_hogar_distribution = client.query(f"""
SELECT
  VP_o_VF_HOGAR,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VP_o_VF_HOGAR
ORDER BY VP_o_VF_HOGAR
""").result().to_dataframe()

vp_o_vf_hogar_distribution.to_csv(
    LOG_DIR / "stage3_vp_o_vf_hogar_distribution.csv",
    index=False
)

display(vp_o_vf_hogar_distribution)

Variables fuente de VP_o_VF_HOGAR verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN VP_HOGAR = 1
      OR VF_HOGAR = 1
    THEN 1

    -- Equivale a:
    -- RECODE VP_o_VF_HOGAR (SYSMIS = 0).
    ELSE 0
  END AS VP_o_VF_HOGAR

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VP_o_VF_HOGAR creado correctamente.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,12618,6189,0,0


VP_o_VF_HOGAR validado: 18,807 filas, dominio 0/1 y correspondencia exacta.


,VP_o_VF_HOGAR,n
0,0,12618
1,1,6189


In [23]:
# ============================================================
# Traducción SPSS 3.2 — VP_VF_HOGAR
# Coocurrencia de violencia psicológica y física en el hogar
#
# Regla:
# VP_VF_HOGAR = 1 cuando:
#   VP_HOGAR = 1 AND VF_HOGAR = 1
# En cualquier otro caso = 0
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

# 1. Verificar variables fuente
existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

required_vp_vf_hogar = [
    "VP_HOGAR",
    "VF_HOGAR",
]

missing_vp_vf_hogar = sorted(
    set(required_vp_vf_hogar) - existing_columns
)

if missing_vp_vf_hogar:
    raise RuntimeError(
        "No se puede crear VP_VF_HOGAR. "
        "Faltan variables fuente: "
        + ", ".join(missing_vp_vf_hogar)
    )

print("Variables fuente de VP_VF_HOGAR verificadas.")


# 2. Preparar reejecución segura
select_prefix = (
    "* EXCEPT(VP_VF_HOGAR)"
    if "VP_VF_HOGAR" in existing_columns
    else "*"
)


# 3. Crear o reemplazar VP_VF_HOGAR
vp_vf_hogar_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN VP_HOGAR = 1
     AND VF_HOGAR = 1
    THEN 1
    ELSE 0
  END AS VP_VF_HOGAR

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_32_vp_vf_hogar.sql").write_text(
    vp_vf_hogar_sql,
    encoding="utf-8"
)

print(vp_vf_hogar_sql)

client.query(vp_vf_hogar_sql).result()

print("VP_VF_HOGAR creado correctamente.")


# 4. Validar universo, dominio y consistencia
vp_vf_hogar_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VP_VF_HOGAR IS NULL
    OR VP_VF_HOGAR NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VP_VF_HOGAR = 0) AS zero_values,

  COUNTIF(VP_VF_HOGAR = 1) AS one_values,

  COUNTIF(
    VP_HOGAR = 1
    AND VF_HOGAR = 1
    AND VP_VF_HOGAR != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    NOT (VP_HOGAR = 1 AND VF_HOGAR = 1)
    AND VP_VF_HOGAR != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vp_vf_hogar_validation.to_csv(
    LOG_DIR / "stage3_vp_vf_hogar_validation.csv",
    index=False
)

display(vp_vf_hogar_validation)

r = vp_vf_hogar_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VP_VF_HOGAR alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VP_VF_HOGAR contiene valores fuera de 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VP_VF_HOGAR no coincide con la lógica esperada."
    )

print(
    "VP_VF_HOGAR validado: "
    "18,807 filas, dominio 0/1 y correspondencia exacta."
)


# 5. Distribución
vp_vf_hogar_distribution = client.query(f"""
SELECT
  VP_VF_HOGAR,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VP_VF_HOGAR
ORDER BY VP_VF_HOGAR
""").result().to_dataframe()

vp_vf_hogar_distribution.to_csv(
    LOG_DIR / "stage3_vp_vf_hogar_distribution.csv",
    index=False
)

display(vp_vf_hogar_distribution)

Variables fuente de VP_VF_HOGAR verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN VP_HOGAR = 1
     AND VF_HOGAR = 1
    THEN 1
    ELSE 0
  END AS VP_VF_HOGAR

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VP_VF_HOGAR creado correctamente.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,16593,2214,0,0


VP_VF_HOGAR validado: 18,807 filas, dominio 0/1 y correspondencia exacta.


,VP_VF_HOGAR,n
0,0,16593
1,1,2214


In [24]:
# ============================================================
# Validación conjunta — Bloque SPSS 3.2 Violencia en el hogar
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

indicators_32 = [
    "VP_HOGAR",
    "VF_HOGAR",
    "VF_HOGAR_01",
    "VF_HOGAR_03",
    "VN_HOGAR1",
    "INDICADOR_8_3_5",
    "VP_o_VF_HOGAR",
    "VP_VF_HOGAR",
]

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_32 = sorted(
    set(indicators_32) - existing_columns
)

if missing_32:
    raise RuntimeError(
        "No se puede cerrar el bloque 3.2. "
        "Faltan indicadores: "
        + ", ".join(missing_32)
    )

validation_parts = []

for variable in indicators_32:
    validation_sql = f"""
    SELECT
      '{variable}' AS variable,
      COUNT(*) AS total_rows,
      COUNTIF(
        `{variable}` IS NULL
        OR `{variable}` NOT IN (0, 1)
      ) AS invalid_values,
      COUNTIF(`{variable}` = 0) AS zero_values,
      COUNTIF(`{variable}` = 1) AS one_values
    FROM `{A}`
    """

    validation_parts.append(
        client.query(validation_sql)
        .result()
        .to_dataframe()
    )

validation_32 = pd.concat(
    validation_parts,
    ignore_index=True
)

validation_32.to_csv(
    LOG_DIR / "stage3_syntax_32_final_validation.csv",
    index=False
)

display(validation_32)

if (validation_32["total_rows"] != EXPECTED_ROWS).any():
    raise RuntimeError(
        "Algún indicador 3.2 alteró el universo analítico."
    )

if (validation_32["invalid_values"] > 0).any():
    raise RuntimeError(
        "Algún indicador 3.2 contiene valores fuera de 0/1."
    )

consistency_32 = client.query(f"""
SELECT
  COUNTIF(
    VN_HOGAR1 !=
    CASE
      WHEN VF_HOGAR_01 = 1 OR VF_HOGAR_03 = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_vn_hogar1,

  COUNTIF(
    INDICADOR_8_3_5 !=
    CASE
      WHEN VP_HOGAR = 1
        OR VF_HOGAR = 1
        OR VF_HOGAR_01 = 1
        OR VF_HOGAR_03 = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_indicador_836,

  COUNTIF(
    VP_o_VF_HOGAR !=
    CASE
      WHEN VP_HOGAR = 1 OR VF_HOGAR = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_vp_o_vf_hogar,

  COUNTIF(
    VP_VF_HOGAR !=
    CASE
      WHEN VP_HOGAR = 1 AND VF_HOGAR = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_vp_vf_hogar

FROM `{A}`
""").result().to_dataframe()

consistency_32.to_csv(
    LOG_DIR / "stage3_syntax_32_consistency_validation.csv",
    index=False
)

display(consistency_32)

if (consistency_32.iloc[0] > 0).any():
    raise RuntimeError(
        "Falló la consistencia interna del bloque 3.2."
    )

print(
    "Bloque SPSS 3.2 validado: "
    "18,807 filas, dominio 0/1 y consistencia interna aprobada."
)

,variable,total_rows,invalid_values,zero_values,one_values
0,VP_HOGAR,18807,0,13538,5269
1,VF_HOGAR,18807,0,15673,3134
2,VF_HOGAR_01,18807,0,18484,323
3,VF_HOGAR_03,18807,0,15037,3770
4,VN_HOGAR1,18807,0,14842,3965
5,INDICADOR_8_3_5,18807,0,10852,7955
6,VP_o_VF_HOGAR,18807,0,12618,6189
7,VP_VF_HOGAR,18807,0,16593,2214


,bad_vn_hogar1,bad_indicador_836,bad_vp_o_vf_hogar,bad_vp_vf_hogar
0,0,0,0,0


Bloque SPSS 3.2 validado: 18,807 filas, dominio 0/1 y consistencia interna aprobada.


## Corrección final de consistencia del diccionario

Materializa `n_formas_justificadas` y valida `INDICADOR_8_3_5`. `DESP` no se fuerza en CRS04 porque su regla fuente depende de `VFNNTV` y corresponde al bloque de desprotección 9–11 años.


In [25]:
# ============================================================
# CORRECCIÓN FINAL DE CONSISTENCIA — Stage 03
#
# Materializa:
#   1. n_formas_justificadas
#
# INDICADOR_8_3_5 ya se crea en el bloque 3.2 anterior
# (corregido desde el nombre erróneo INDICADOR_8_3_6).
#
# DESP NO se materializa aquí:
# la sintaxis fuente identificada para DESP usa VFNNTV y corresponde
# al bloque de desprotección 9–11 años, por lo que no debe forzarse
# dentro del analytical CRS04 de adolescentes 12–17.
#
# Reejecutable: usa SELECT * EXCEPT(...) si la columna ya existe.
# ============================================================

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

table_before_patch = client.get_table(A)

existing_columns = {
    field.name
    for field in table_before_patch.schema
}

required_patch_sources = [
    "justifica_castigo_docente",
    "justifica_castigo_parental",
    "INDICADOR_8_3_5",
]

missing_patch_sources = sorted(
    set(required_patch_sources) - existing_columns
)

if missing_patch_sources:
    raise RuntimeError(
        "No se puede aplicar la corrección final. "
        "Faltan variables fuente: "
        + ", ".join(missing_patch_sources)
    )

if "n_formas_justificadas" in existing_columns:
    source_select = "* EXCEPT(`n_formas_justificadas`)"
else:
    source_select = "*"

sql_final_consistency_patch = f"""
CREATE OR REPLACE TABLE `{A}` AS
SELECT
  {source_select},

  CASE
    WHEN justifica_castigo_docente IS NULL
     AND justifica_castigo_parental IS NULL
    THEN NULL
    ELSE
      COALESCE(justifica_castigo_docente, 0)
      + COALESCE(justifica_castigo_parental, 0)
  END AS n_formas_justificadas

FROM `{A}`
"""

(SQL_DIR / "stage3_final_consistency_patch.sql").write_text(
    sql_final_consistency_patch,
    encoding="utf-8",
)

print(sql_final_consistency_patch)

client.query(
    sql_final_consistency_patch,
    location=LOCATION,
).result()

validation_final_patch = client.query(
    f"""
    SELECT
      COUNT(*) AS total_rows,

      COUNTIF(
        n_formas_justificadas IS NOT NULL
        AND n_formas_justificadas NOT IN (0, 1, 2)
      ) AS invalid_n_formas,

      COUNTIF(
        INDICADOR_8_3_5 IS NOT NULL
        AND INDICADOR_8_3_5 NOT IN (0, 1)
      ) AS invalid_indicador_8_3_5

    FROM `{A}`
    """,
    location=LOCATION,
).result().to_dataframe()

display(validation_final_patch)

validation_final_patch.to_csv(
    LOG_DIR / "stage3_final_consistency_patch_validation.csv",
    index=False,
)

r = validation_final_patch.iloc[0]

if int(r["total_rows"]) != EXPECTED_ROWS:
    raise RuntimeError(
        "La corrección alteró el universo de 18,807 filas."
    )

if int(r["invalid_n_formas"]) != 0:
    raise RuntimeError(
        "n_formas_justificadas contiene valores fuera de 0,1,2,NULL."
    )

if int(r["invalid_indicador_8_3_5"]) != 0:
    raise RuntimeError(
        "INDICADOR_8_3_5 contiene valores fuera de 0,1,NULL."
    )

print(
    "PASS: n_formas_justificadas e INDICADOR_8_3_5 "
    "quedan materializados y validados."
)



CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS
SELECT
  *,

  CASE
    WHEN justifica_castigo_docente IS NULL
     AND justifica_castigo_parental IS NULL
    THEN NULL
    ELSE
      COALESCE(justifica_castigo_docente, 0)
      + COALESCE(justifica_castigo_parental, 0)
  END AS n_formas_justificadas

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`



,total_rows,invalid_n_formas,invalid_indicador_8_3_5
0,18807,0,0


PASS: n_formas_justificadas e INDICADOR_8_3_5 quedan materializados y validados.
